<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_8_%D0%90%D0%B2%D1%82%D0%BE%D0%BD%D0%BE%D0%BC%D0%BD%D1%8B%D0%B5_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D1%8B_%D0%BF%D0%BB%D0%B0%D0%BD%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B8_%D1%81%D0%B0%D0%BC%D0%BE%D0%BE%D1%86%D0%B5%D0%BD%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.8. Автономные агенты: планирование и самооценка

## Введение: от реактивных помощников к проактивным агентам

Поздравляю! Мы прошли невероятный путь. В Лекции 6.1 мы создавали простого RAG-агента, который отвечал на вопросы по документам. В Лекции 6.4 мы научили агента вызывать инструменты и принимать решения. В Лекции 6.6 мы построили команду специализированных агентов, а в Лекции 6.7 добавили супервайзера, который управляет этой командой. Но все эти агенты были **реактивными** – они получали один вопрос, выполняли несколько шагов и останавливались. Они не планировали долгосрочные действия, не оценивали свой прогресс и не корректировали план на ходу.

Теперь мы подходим к **вершине** – автономным агентам. Представьте, что вы даёте агенту не вопрос, а **цель**: «Напиши аналитический отчёт о состоянии рынка ИИ в 2026 году». Агент сам:
1. Разбивает задачу на подзадачи (собрать данные, проанализировать, написать текст, отредактировать).
2. Выполняет их последовательно, используя доступные инструменты.
3. Проверяет качество своей работы.
4. Если результат неудовлетворительный – возвращается и исправляет ошибки.

Это и есть **автономный агент** – система, которая самостоятельно планирует, действует, оценивает и адаптируется до тех пор, пока цель не будет достигнута. Это уже не просто «ответ на вопрос», а настоящий цифровой помощник, способный решать сложные многошаговые задачи.

В этой лекции мы реализуем такого агента. Мы объединим три ключевых паттерна:
- **ReAct (Reasoning + Acting)** – агент чередует размышления и действия.
- **Plan‑and‑Execute** – агент сначала составляет план, потом выполняет.
- **Reflexion** – агент оценивает свои результаты и учится на ошибках.

К концу лекции вы получите полностью автономного агента, который сможет самостоятельно выполнять сложные задачи – от написания отчётов до проведения исследований. Поехали!

---

## Тема 1. Что такое автономный агент и зачем он нужен

### 1.1. Отличие от реактивного агента

Все агенты, которые мы строили ранее, были **реактивными**. Они получали запрос и выполняли заранее определённую последовательность действий. Даже супервайзер из Лекции 6.7, хотя и принимал решения на каждом шаге, делал это в рамках одной задачи. Как только ответ был сгенерирован – работа завершалась.

**Автономный агент** работает иначе:

| Аспект | Реактивный агент | Автономный агент |
|--------|------------------|------------------|
| **Инициатива** | Отвечает на запрос | Самостоятельно ставит подцели |
| **Планирование** | Фиксированный порядок шагов | Динамический план, который может меняться |
| **Оценка** | Не проверяет качество | Проверяет результат и исправляет ошибки |
| **Остановка** | Останавливается после ответа | Работает до достижения цели |
| **Адаптивность** | Не меняет стратегию | Меняет план при неудачах |

**Пример:** если реактивному агенту сказать «напиши отчёт», он может просто сгенерировать текст на основе имеющихся данных. Автономный агент сначала подумает: «Что нужно для отчёта? Какие данные собрать? Где их взять? Проверить ли факты?» – и только потом начнёт действовать, постоянно оценивая прогресс.

### 1.2. Примеры задач для автономных агентов

Автономные агенты особенно полезны там, где задача не может быть решена за один шаг:

- **Написание аналитического отчёта** – сбор данных, анализ, структурирование, написание, редактура.
- **Исследование темы** – поиск информации, проверка источников, синтез знаний, формулировка выводов.
- **Автоматизация рутины** – обработка писем, планирование встреч, управление проектами.
- **Обучение и саморазвитие** – агент изучает новую тему и проверяет свои знания.
- **Программирование** – написание кода, тестирование, отладка, рефакторинг.

В каждом из этих сценариев агенту нужно не просто ответить, а **достичь цели** – и для этого требуется планирование, оценка и адаптация.

### 1.3. Основные компоненты автономного агента

Автономный агент состоит из трёх ключевых компонентов, работающих в цикле:

1. **Планировщик (Planner)** – разбивает цель на подзадачи и определяет порядок их выполнения.
2. **Исполнитель (Executor)** – выполняет подзадачи, используя доступные инструменты.
3. **Оценщик (Evaluator / Self‑Critic)** – анализирует результат, проверяет, достигнута ли цель, и решает, нужно ли корректировать план.

К этим трём добавляется **память** – не только краткосрочная (история диалога), но и долгосрочная (сохранение результатов предыдущих шагов, чтобы не повторять их).

**Схема работы:**

```
Пользователь задаёт цель
        ↓
┌───────────────────────────────────────┐
│  Планировщик (LLM)                    │
│  "Что нужно сделать для достижения?   │
│   Какой следующий шаг?"               │
└──────────────────┬────────────────────┘
                   ↓
┌───────────────────────────────────────┐
│  Исполнитель (Executor)               │
│  Выполняет шаг (вызов инструмента,    │
│  поиск, вычисление)                   │
└──────────────────┬────────────────────┘
                   ↓
┌───────────────────────────────────────┐
│  Оценщик (Self‑Critic)                │
│  "Достигнута ли цель? Нужно ли        │
│   изменить план?"                     │
└──────────────────┬────────────────────┘
                   ↓
        ┌──────────┴──────────┐
        │  Цель достигнута?    │
        │  Да → Ответ          │
        │  Нет → Вернуться к   │
        │        планировщику  │
        └─────────────────────┘
```

Каждый из этих компонентов может быть реализован как отдельный LLM-агент или как один агент, который выполняет все три роли в цикле.

### 1.4. Обзор подходов к автономности

Существует несколько популярных паттернов для построения автономных агентов:

#### ReAct (Reasoning + Acting)

Агент чередует «размышление» и «действие». На каждом шаге он пишет, что собирается сделать, выполняет действие, анализирует результат и решает, что делать дальше.

```
Шаг 1: "Мне нужно найти информацию о компании X" → поиск
Шаг 2: "Теперь нужно сравнить с компанией Y" → поиск
Шаг 3: "Данные собраны, можно писать ответ" → генерация
```

Этот паттерн мы уже использовали в Лекции 6.4 – агент с инструментами по сути работал по схеме ReAct.

#### Plan‑and‑Execute

Агент сначала составляет полный план действий, а затем последовательно его выполняет. Это делает поведение более предсказуемым и позволяет видеть весь маршрут заранее.

```
План:
1. Найти информацию о компании X
2. Найти информацию о компании Y
3. Сравнить показатели
4. Написать отчёт
5. Проверить факты
→ Выполнение по шагам
```

#### Reflexion (Самооценка)

Агент генерирует ответ, затем критикует его, указывает на ошибки и генерирует исправленный вариант. Этот цикл повторяется, пока качество не станет удовлетворительным.

```
Шаг 1: "Вот мой ответ..." (генерация)
Шаг 2: "Я ошибся в датах, нужно исправить" (рефлексия)
Шаг 3: "Вот исправленный ответ" (новая генерация)
```

#### Tree of Thoughts (Дерево мыслей)

Агент генерирует несколько вариантов решения, оценивает каждый и выбирает лучший. Это похоже на то, как человек рассматривает разные варианты перед принятием решения.

В этой лекции мы объединим **Plan‑and‑Execute** и **Reflexion**, чтобы получить максимально автономного агента.

### 1.5. Какие пакеты нужны

Для реализации автономного агента нам не понадобятся новые библиотеки. Всё, что мы использовали раньше, остаётся:

```bash
pip install langchain langchain-ollama langgraph chromadb sentence-transformers
```

Мы будем использовать:
- **LangGraph** – для построения графа с циклом (планировщик → исполнитель → оценщик → планировщик).
- **LangChain** – для работы с LLM и инструментами.
- **Ollama** – как локальный сервер для LLM.
- **Chroma** – для долгосрочной памяти (сохранение результатов шагов).

Никаких дополнительных установок не требуется – всё уже есть в вашем проекте.

---

## Краткий итог Тема 1

- **Автономный агент** – это система, которая самостоятельно планирует, действует, оценивает и адаптируется до достижения цели.
- Он отличается от реактивного агента **проактивностью** – он не просто отвечает, а сам ставит подцели и корректирует стратегию.
- **Три ключевых компонента**: планировщик, исполнитель, оценщик.
- **Основные паттерны**: ReAct, Plan‑and‑Execute, Reflexion, Tree of Thoughts.
- Мы будем использовать **LangGraph** для реализации цикла и **Ollama** для LLM.

---

**В следующей теме мы перейдём к реализации планировщика и настроим цикл «план → действие → оценка».**

## Тема 2. Паттерн ReAct (Reasoning + Acting) (скрипт `react_agent.py`)

В предыдущей лекции мы говорили о том, что автономный агент должен уметь планировать, действовать и оценивать результат. Самый фундаментальный паттерн, на котором строится большинство автономных агентов, – это **ReAct (Reasoning + Acting)**. Мы уже неявно использовали его в Лекции 6.4, когда агент с инструментами решал, что вызывать, и выполнял действия. Но тогда мы не выделяли «мысли» как отдельный элемент. Теперь мы сделаем это осознанно и структурированно, добавив реальные инструменты и улучшенную наблюдаемость.

> **📦 Важно!** Для работы веб-поиска через DuckDuckGo установите дополнительный пакет:
> ```bash
> pip install duckduckgo-search
> ```
> Он не требует API-ключа и работает "из коробки".

---

### 2.1. Идея: чередовать «подумать» и «сделать»

ReAct предлагает простую, но мощную идею: агент на каждом шаге сначала **думает** (рассуждает), затем **действует**, а потом **наблюдает** результат. Модель явно пишет свои мысли, что делает процесс **прозрачным** и **отлаживаемым**.

**Цикл ReAct:**

```
Пользователь: "Напиши отчёт о компании X"
Агент: Мысль: нужно найти информацию о компании X.
       Действие: search_docs("X")
Наблюдение: [результаты поиска]
Агент: Мысль: теперь нужно собрать финансовые показатели.
       Действие: search_docs("финансовые показатели X")
Наблюдение: [результаты поиска]
Агент: Мысль: данных достаточно, можно написать ответ.
       (нет действия, завершаем)
```

**Преимущества:**

- **Прозрачность** – мы видим, почему агент делает то или иное действие.
- **Отладка** – легко понять, на каком шаге произошла ошибка.
- **Гибкость** – агент может передумать, если наблюдение не совпадает с ожиданиями.

---

### 2.2. Формат промпта

Для реализации ReAct мы будем использовать следующий формат промпта:

```
Мысль: {мысль агента}
Действие: {имя инструмента, параметры}
Наблюдение: {результат выполнения инструмента}
Мысль: {следующая мысль}
...
```

В нашей реализации мы используем `bind_tools` для вызова инструментов и храним наблюдения в отдельном поле состояния, что позволяет агенту анализировать предыдущие шаги.

**Системный промпт для ReAct:**

```python
SYSTEM_PROMPT = """
Ты — автономный агент, работающий по методологии ReAct (Reasoning + Acting).

Твоя задача — достичь цели пользователя, используя инструменты. Всегда следуй этому циклу:

1. **Мысль** (Reasoning) – проанализируй, что известно, и что нужно сделать дальше.
2. **Действие** (Acting) – если нужна дополнительная информация, **вызови подходящий инструмент** (search_docs, web_search или calculate). Никогда не пиши действие текстом — всегда используй вызов инструмента.
3. **Наблюдение** (Observation) – после получения результата инструмента, **обязательно** проанализируй его текстом. Скажи, что ты узнал и достаточно ли этого для ответа.
4. Если цель ещё не достигнута, вернись к шагу 1 (новая Мысль).
5. Если цель достигнута, напиши **финальный ответ** пользователю (без вызова инструментов).

Важно:
- Не завершай работу, пока не будет достаточно информации для полного и точного ответа.
- Если результат инструмента пуст или не содержит нужных данных, попробуй другой инструмент или переформулируй запрос.
- В финальном ответе суммируй все наблюдения и дай чёткий, структурированный ответ.
"""
```

---

### 2.3. Реализация на LangGraph

Создадим граф с двумя узлами:

1. **`agent`** – вызывает LLM с инструментами, получает мысль и действие.
2. **`tools`** – выполняет действие, возвращает наблюдение.

**Цикл:**

```
agent → tools → agent → tools → ... → завершение
```

**Состояние** содержит:
- `messages` – история сообщений (включая мысли, действия, наблюдения).
- `iteration` – счётчик шагов.
- `question` – исходная цель.
- `observations` – список наблюдений для анализа.

---

### 2.4. Инструменты: реальный поиск и вычисления

В нашей реализации мы используем три инструмента:

1. **`search_docs`** – поиск в локальной базе знаний (заглушка для демонстрации).
2. **`web_search`** – реальный поиск через DuckDuckGo (без API-ключа).  
   > Для работы установите пакет: `pip install duckduckgo-search`
3. **`calculate`** – безопасное выполнение математических вычислений.

```python
from langchain_community.tools import DuckDuckGoSearchRun

web_search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Выполняет поиск в интернете с помощью DuckDuckGo."""
    try:
        results = web_search_tool.invoke(query)
        return results[:1000] if len(results) > 1000 else results
    except Exception as e:
        return f"Ошибка поиска: {e}"
```

---

### 2.5. Полный код `react_agent.py`

Ниже представлен полный код агента с реализацией паттерна ReAct. Он включает реальный веб-поиск через DuckDuckGo, хранение наблюдений, улучшенный системный промпт и защиту от бесконечного цикла.

**Перед запуском убедитесь, что установлены все зависимости:**

```bash
pip install langchain langchain-ollama langgraph langchain-community duckduckgo-search
```

```python
"""
react_agent.py - Реализация паттерна ReAct на LangGraph (улучшенная версия)
Лекция 6.8, Тема 2

Особенности:
- Реальный веб-поиск через DuckDuckGo
- Хранение наблюдений для анализа
- Улучшенный системный промпт
- Защита от бесконечного цикла
"""

import math
import re
from typing import TypedDict, List, Annotated, Literal, Optional
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun  # реальный поиск

# ============================================================================
# 1. ИНСТРУМЕНТЫ
# ============================================================================

@tool
def search_docs(query: str) -> str:
    """Ищет информацию в локальной базе знаний."""
    # Для демонстрации оставляем заглушку, но можно заменить на реальный ретривер
    if "RAG" in query.upper():
        return "RAG (Retrieval-Augmented Generation) — это подход, сочетающий поиск и генерацию. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM)."
    elif "LLM" in query.upper():
        return "Большие языковые модели (LLM) обучаются на больших объёмах текстов."
    else:
        return "Информация не найдена."

# Реальный веб-поиск (без API-ключа)
web_search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Выполняет поиск в интернете с помощью DuckDuckGo."""
    try:
        results = web_search_tool.invoke(query)
        # Ограничим длину, чтобы не перегружать контекст
        return results[:1000] if len(results) > 1000 else results
    except Exception as e:
        return f"Ошибка поиска: {e}"

@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления (безопасно)."""
    # Разрешаем только числа и простые операторы
    if not re.match(r'^[\d+\-*/().\s]+$', expression):
        return "Ошибка: недопустимые символы в выражении."
    try:
        result = eval(expression, {"__builtins__": {}})
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

tools = [search_docs, web_search, calculate]

# ============================================================================
# 2. НАСТРОЙКА LLM
# ============================================================================

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=512)
llm_with_tools = llm.bind_tools(tools)

# Улучшенный системный промпт
SYSTEM_PROMPT = """
Ты — автономный агент, работающий по методологии ReAct (Reasoning + Acting).

Твоя задача — достичь цели пользователя, используя инструменты. Всегда следуй этому циклу:

1. **Мысль** (Reasoning) – проанализируй, что известно, и что нужно сделать дальше.
2. **Действие** (Acting) – если нужна дополнительная информация, **вызови подходящий инструмент** (search_docs, web_search или calculate). Никогда не пиши действие текстом — всегда используй вызов инструмента.
3. **Наблюдение** (Observation) – после получения результата инструмента, **обязательно** проанализируй его текстом. Скажи, что ты узнал и достаточно ли этого для ответа.
4. Если цель ещё не достигнута, вернись к шагу 1 (новая Мысль).
5. Если цель достигнута, напиши **финальный ответ** пользователю (без вызова инструментов).

Важно:
- Не завершай работу, пока не будет достаточно информации для полного и точного ответа.
- Если результат инструмента пуст или не содержит нужных данных, попробуй другой инструмент или переформулируй запрос.
- В финальном ответе суммируй все наблюдения и дай чёткий, структурированный ответ.

Инструменты:
- search_docs(query) – поиск в локальной базе знаний (ограниченная информация).
- web_search(query) – поиск в интернете (актуальные данные).
- calculate(expression) – вычисление математических выражений.

Начинай!
"""

# ============================================================================
# 3. СОСТОЯНИЕ (добавили observations)
# ============================================================================

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    iteration: int
    max_iterations: int
    observations: List[str]   # храним результаты наблюдений

# ============================================================================
# 4. УЗЛЫ ГРАФА
# ============================================================================

def agent_node(state: AgentState) -> dict:
    iteration = state.get("iteration", 0) + 1
    print(f"\n🧠 Итерация {iteration}")

    if iteration > state.get("max_iterations", 10):
        print("⚠️  Превышен лимит итераций.")
        return {
            "messages": [AIMessage(content="Не удалось достичь цели за отведённое время.")],
            "iteration": iteration
        }

    messages = state["messages"]
    # Добавляем системный промпт, если его нет
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    # Вставляем напоминание о наблюдениях в контекст, если они есть
    observations = state.get("observations", [])
    if observations:
        obs_text = "\n".join([f"Наблюдение {i+1}: {obs}" for i, obs in enumerate(observations)])
        reminder = f"\n\nТвои предыдущие наблюдения:\n{obs_text}\n\nТеперь, основываясь на них, реши, что делать дальше."
        messages.append(HumanMessage(content=reminder))

    response = llm_with_tools.invoke(messages)
    print(f"💬 Ответ LLM: {response.content[:150]}..." if response.content else "💬 Ответ LLM: (пусто)")

    if hasattr(response, "tool_calls") and response.tool_calls:
        print(f"🔧 Вызваны инструменты: {[tc['name'] for tc in response.tool_calls]}")
    else:
        print("✅ Инструменты не вызваны, возможно финальный ответ.")

    return {"messages": [response], "iteration": iteration}

def tools_node(state: AgentState) -> dict:
    last_message = state["messages"][-1]
    tool_calls = last_message.tool_calls

    if not tool_calls:
        return {"messages": []}

    tool_messages = []
    observations = state.get("observations", [])

    for tc in tool_calls:
        tool_name = tc["name"]
        tool_args = tc["args"]
        tool_map = {t.name: t for t in tools}
        if tool_name in tool_map:
            result = tool_map[tool_name].invoke(tool_args)
            print(f"🔧 Инструмент {tool_name} вернул: {result[:100]}...")
            # Сохраняем наблюдение
            observations.append(f"{tool_name}({tool_args}) -> {result}")
            tool_messages.append(
                ToolMessage(content=str(result), tool_call_id=tc["id"])
            )
        else:
            err_msg = f"Инструмент {tool_name} не найден."
            observations.append(err_msg)
            tool_messages.append(
                ToolMessage(content=err_msg, tool_call_id=tc["id"])
            )

    return {"messages": tool_messages, "observations": observations}

# ============================================================================
# 5. МАРШРУТИЗАЦИЯ
# ============================================================================

def route_after_agent(state: AgentState) -> Literal["tools", "finish"]:
    last_message = state["messages"][-1]
    # Если есть вызовы инструментов — идём в tools
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    # Если инструменты не вызваны, считаем, что это финальный ответ
    return "finish"

# ============================================================================
# 6. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tools_node)

builder.set_entry_point("agent")
builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {
        "tools": "tools",
        "finish": END
    }
)
builder.add_edge("tools", "agent")

graph = builder.compile()

# ============================================================================
# 7. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    questions = [
        "Что такое RAG и как он работает?",
        "Сколько будет 25% от 200?",
        "Сравни RAG и обычный ChatGPT.",
        "Какая сегодня погода в Москве?"  # проверим реальный поиск
    ]

    for q in questions:
        print("\n" + "=" * 60)
        print(f"📝 Вопрос: {q}")
        print("=" * 60)

        initial_state = {
            "messages": [HumanMessage(content=q)],
            "question": q,
            "iteration": 0,
            "max_iterations": 10,   # увеличено
            "observations": []
        }

        try:
            result = graph.invoke(initial_state, config={"recursion_limit": 15})
        except Exception as e:
            print(f"❌ Ошибка выполнения: {e}")
            continue

        # Извлекаем финальный ответ (последнее AIMessage без tool_calls)
        final_answer = None
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and not (hasattr(msg, "tool_calls") and msg.tool_calls):
                final_answer = msg.content
                break

        print(f"\n✅ Финальный ответ:\n{final_answer if final_answer else 'Не сгенерирован'}")
        print(f"📊 Шагов выполнено: {result.get('iteration', 0)}")
        print(f"📋 Наблюдений: {len(result.get('observations', []))}")
        for i, obs in enumerate(result.get('observations', []), 1):
            print(f"   {i}. {obs[:150]}...")
```

---

### 2.6. Результаты тестирования

При запуске на четырёх вопросах агент показал:

| Вопрос | Действие | Результат |
|--------|----------|-----------|
| «Что такое RAG?» | 2 поиска, затем ответ | Агент собрал информацию и сгенерировал ответ |
| «25% от 200?» | 1 вычисление, затем ответ | Агент правильно вычислил и ответил |
| «Сравни RAG и ChatGPT» | 2 поиска, остановился на мысли | Требует доработки (Reflexion) |
| «Погода в Москве» | 2 одинаковых поиска, остановился | Требует доработки (Reflexion) |

**Что работает хорошо:**
- Агент правильно выбирает инструменты.
- Сохраняет наблюдения и использует их.
- Останавливается при достижении цели.

**Что требует улучшения:**
- Сложные вопросы требуют самооценки (Reflexion).
- Агент иногда повторяет одинаковые действия.
- Не всегда чётко отделяет финальный ответ от мыслей.

Эти проблемы будут решены в следующей теме – **Reflexion (самооценка и рефлексия)**.

---

## Краткий итог Тема 2

- **ReAct** – фундаментальный паттерн, где агент чередует «мысли» и «действия».
- Мы реализовали агента с реальным веб-поиском через DuckDuckGo (не забудьте установить `duckduckgo-search`).
- Агент сохраняет наблюдения и использует их для принятия решений.
- Цикл продолжается до тех пор, пока агент не перестанет вызывать инструменты или не достигнет лимита итераций.
- Для простых задач агент работает отлично, для сложных требуется самооценка.

---

**В следующей теме мы добавим самооценку и рефлексию – научим агента проверять свои ответы и исправлять ошибки.**

## Тема 3. Планировщик (Plan‑and‑Execute)

В прошлой теме мы построили агента на основе **ReAct** – он отлично справляется с короткими задачами, где нужно сделать 2–3 шага. Но что, если задача требует десятков действий? Например: «Проанализируй рынок ИИ, собери данные по пяти компаниям, сравни их показатели, напиши отчёт и отредактируй его». Такой процесс может включать 15–20 шагов. В режиме ReAct агент принимает решение «на ходу», что часто приводит к:
- **потере контекста** – чем длиннее диалог, тем сложнее модели удерживать все детали;
- **повторным действиям** – агент может забыть, что уже искал информацию, и повторить поиск;
- **сложности оценки прогресса** – непонятно, сколько ещё шагов осталось.

Выход – паттерн **Plan‑and‑Execute**. Вместо того чтобы думать на каждом шаге, агент сначала составляет **полный план** действий, а затем **последовательно выполняет** его, при необходимости корректируя. Это похоже на то, как человек пишет список дел на день и вычёркивает пункты.

---

### 3.1. Отличие от ReAct

| Аспект | ReAct | Plan‑and‑Execute |
|--------|-------|------------------|
| **Планирование** | Пошаговое, принимается на каждой итерации | Сначала генерируется полный план |
| **Прозрачность** | Видны только текущие мысли | Весь маршрут виден заранее |
| **Адаптивность** | Может менять решение на любом шаге | Изменения требуют перепланирования |
| **Параллелизм** | Только последовательно | Независимые шаги можно выполнить параллельно |
| **Сложность** | Хорошо для < 5 шагов | Хорошо для ≥ 10 шагов |

**Ключевая идея** – мы разделяем **проектирование** (планирование) и **исполнение**. Это упрощает отладку: если план не сработал, мы можем пересмотреть только его, не перезапуская весь процесс.

---

### 3.2. Узлы графа

В нашей реализации будет три основных узла, соединённых в цикл:

1. **`planner`** – генерирует начальный план (или перепланирует, если текущий провалился). План – это список шагов с описанием и, при необходимости, указанием инструмента.

2. **`executor`** – выполняет текущий шаг. Если шаг требует инструмента – вызывает его; если это чисто текстовый шаг (например, «сформулировать вывод») – использует LLM без инструментов. Результат сохраняется.

3. **`replanner`** – анализирует результат выполнения шага. Если шаг завершился ошибкой или результат не соответствует ожиданиям, replanner корректирует план (изменяет текущий шаг, добавляет новые шаги, меняет порядок) и возвращает управление планировщику.

Цикл:
```
planner → executor → (если успех) → следующий шаг → executor ...
                ↓ (если ошибка)
             replanner → planner (с новым планом)
```

**Важно**: мы не просто завершаемся при ошибке, а **перепланируем**, то есть учимся на неудачах.

---

### 3.3. Состояние агента

Состояние содержит не только историю сообщений, но и структурированную информацию о плане:

```python
class PlanExecuteState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]   # диалог
    question: str                                          # исходная цель
    plan: List[dict]                                       # список шагов, каждый с полями:
                                                           #   - step_number: int
                                                           #   - description: str
                                                           #   - tool: Optional[str] (имя инструмента)
                                                           #   - dependencies: List[int] (номера шагов, от которых зависит)
    current_step: int                                      # индекс текущего шага (0-based)
    step_results: List[dict]                               # результаты выполнения каждого шага
                                                           #   - step_number, result, success, error
    plan_status: Literal["active", "completed", "failed"]
    iteration: int
    max_iterations: int
```

---

### 3.4. Реализация

#### Генерация плана (Planner)

Планировщик получает цель пользователя и генерирует структурированный план в формате JSON. Мы используем системный промпт, который требует выдать массив шагов.

**Пример промпта для планировщика**:

```
Ты – планировщик. Твоя задача – разбить цель пользователя на последовательность шагов.
Каждый шаг должен быть описан понятно и, если необходимо, указать инструмент для выполнения.
Доступные инструменты: search_docs, web_search, calculate.
Если шаг не требует инструмента, поле "tool" оставь пустым.

Ответ должен быть в формате JSON-массива, где каждый элемент имеет поля:
- "step_number": номер шага (начиная с 1)
- "description": описание шага
- "tool": имя инструмента или null
- "dependencies": массив номеров шагов, от которых зависит этот шаг (если нет, то [])

Пример для задачи "Сравни RAG и ChatGPT":
[
  {"step_number": 1, "description": "Найти информацию о RAG", "tool": "web_search", "dependencies": []},
  {"step_number": 2, "description": "Найти информацию о ChatGPT", "tool": "web_search", "dependencies": []},
  {"step_number": 3, "description": "Сравнить основные характеристики", "tool": null, "dependencies": [1,2]},
  {"step_number": 4, "description": "Сформулировать итоговый ответ", "tool": null, "dependencies": [3]}
]
```

Планировщик может быть отдельным LLM-вызовом. В нашем графе он вызывается один раз в начале (или при перепланировании). Мы сохраняем план в состоянии.

#### Исполнитель (Executor)

Исполнитель берёт текущий шаг (по индексу `current_step`) и выполняет его:

- Если у шага есть `tool`, вызываем соответствующий инструмент с параметрами (параметры берутся из описания шага или из контекста).
- Если инструмент не указан, то мы просто просим LLM выполнить шаг (сгенерировать текст на основе предыдущих результатов).

Результат сохраняется в `step_results`. Если инструмент вернул ошибку – помечаем шаг как неудачный.

#### Перепланировщик (Replanner)

Если шаг завершился ошибкой или результат неудовлетворительный (например, пустой ответ), вызывается replanner. Он получает текущий план, результаты выполненных шагов и описание ошибки. Затем он генерирует **скорректированный план** – может изменить текущий шаг, добавить новые шаги для сбора дополнительной информации, или вообще пересмотреть порядок.

Replanner – это тоже LLM, но с другим системным промптом, где модель анализирует причины неудачи и предлагает новый план.

**Важно**: чтобы не зациклиться, мы ограничиваем количество перепланирований (например, не более 3 раз).

---

### 3.5. Преимущества Plan‑and‑Execute

1. **Прозрачность** – пользователь видит весь план заранее и может оценить, насколько адекватно агент понял задачу.
2. **Возможность параллельного выполнения** – шаги без зависимостей можно выполнять одновременно, что ускоряет работу (в нашей реализации мы пока делаем последовательно, но архитектура это позволяет).
3. **Устойчивость к ошибкам** – агент не падает при первой неудаче, а пытается перепланировать.
4. **Контроль качества** – можно проверять промежуточные результаты и прерывать выполнение, если план уводит в сторону.

---

### 3.6. Полный код `plan_execute_agent.py`

Ниже представлена реализация агента Plan‑and‑Execute с использованием LangGraph. Код включает:

- Генерацию плана через LLM.
- Выполнение шагов с вызовом инструментов (search_docs, web_search, calculate).
- Простую логику перепланирования (если инструмент вернул ошибку или пустой результат).
- Защиту от бесконечного цикла.

**Обратите внимание**: для простоты мы не реализуем полноценный replanner с LLM, а используем эвристику: при ошибке мы добавляем новый шаг "уточнить информацию" и продолжаем. В реальных системах replanner тоже может быть LLM-агентом.

```python
"""
plan_execute_agent.py - Реализация паттерна Plan-and-Execute на LangGraph
Лекция 6.8, Тема 3

Особенности:
- Генерация полного плана перед выполнением
- Последовательное выполнение шагов
- Перепланирование при ошибках (упрощённое)
- Использование тех же инструментов, что и в ReAct
"""

import json
import re
from typing import TypedDict, List, Annotated, Literal, Optional, Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

# ============================================================================
# 1. ИНСТРУМЕНТЫ (те же, что и в react_agent.py)
# ============================================================================

@tool
def search_docs(query: str) -> str:
    """Ищет информацию в локальной базе знаний."""
    if "RAG" in query.upper():
        return "RAG (Retrieval-Augmented Generation) — это подход, сочетающий поиск и генерацию. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM)."
    elif "LLM" in query.upper():
        return "Большие языковые модели (LLM) обучаются на больших объёмах текстов."
    else:
        return "Информация не найдена."

web_search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Выполняет поиск в интернете с помощью DuckDuckGo."""
    try:
        results = web_search_tool.invoke(query)
        return results[:1000] if len(results) > 1000 else results
    except Exception as e:
        return f"Ошибка поиска: {e}"

@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления (безопасно)."""
    if not re.match(r'^[\d+\-*/().\s]+$', expression):
        return "Ошибка: недопустимые символы в выражении."
    try:
        result = eval(expression, {"__builtins__": {}})
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

tools = [search_docs, web_search, calculate]
tool_map = {t.name: t for t in tools}

# ============================================================================
# 2. НАСТРОЙКА LLM
# ============================================================================

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=512)

# ============================================================================
# 3. СОСТОЯНИЕ
# ============================================================================

class PlanExecuteState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    plan: List[Dict[str, Any]]          # список шагов
    current_step: int                   # индекс текущего шага (0-based)
    step_results: List[Dict[str, Any]]  # результаты выполнения
    plan_status: Literal["active", "completed", "failed"]
    iteration: int
    max_iterations: int
    replan_count: int                   # счётчик перепланирований

# ============================================================================
# 4. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================================

def parse_plan(response_content: str) -> List[Dict[str, Any]]:
    """Извлекает JSON-план из ответа LLM."""
    try:
        # Ищем блок JSON
        json_match = re.search(r'\[.*\]', response_content, re.DOTALL)
        if json_match:
            plan = json.loads(json_match.group())
            if isinstance(plan, list) and all(isinstance(item, dict) for item in plan):
                return plan
    except Exception as e:
        print(f"Ошибка парсинга плана: {e}")
    # Если не удалось, возвращаем план по умолчанию из одного шага
    return [{"step_number": 1, "description": "Ответить на вопрос", "tool": None, "dependencies": []}]

# ============================================================================
# 5. УЗЛЫ ГРАФА
# ============================================================================

def planner_node(state: PlanExecuteState) -> dict:
    """Генерирует начальный план или перепланирует при необходимости."""
    print("\n📋 Генерация плана...")
    
    # Если план уже есть и мы перепланируем, используем другой промпт
    if state.get("plan") and state.get("replan_count", 0) > 0:
        # Упрощённое перепланирование: просто добавляем шаг "Уточнить информацию"
        # В реальном проекте здесь был бы отдельный LLM-вызов
        plan = state["plan"]
        # Добавляем новый шаг после текущего
        new_step = {
            "step_number": len(plan) + 1,
            "description": "Уточнить информацию с помощью web_search",
            "tool": "web_search",
            "dependencies": [state["current_step"] + 1]  # зависит от предыдущего
        }
        plan.append(new_step)
        print("🔄 План скорректирован (добавлен уточняющий шаг).")
        return {"plan": plan, "replan_count": state.get("replan_count", 0) + 1}
    
    # Генерация нового плана
    system_prompt = """
Ты – планировщик. Твоя задача – разбить цель пользователя на последовательность шагов.
Каждый шаг должен быть описан понятно и, если необходимо, указать инструмент для выполнения.
Доступные инструменты: search_docs, web_search, calculate.
Если шаг не требует инструмента, поле "tool" оставь пустым (null).

Ответ должен быть в формате JSON-массива, где каждый элемент имеет поля:
- "step_number": номер шага (начиная с 1)
- "description": описание шага
- "tool": имя инструмента или null
- "dependencies": массив номеров шагов, от которых зависит этот шаг (если нет, то [])

Пример для задачи "Сравни RAG и ChatGPT":
[
  {"step_number": 1, "description": "Найти информацию о RAG", "tool": "web_search", "dependencies": []},
  {"step_number": 2, "description": "Найти информацию о ChatGPT", "tool": "web_search", "dependencies": []},
  {"step_number": 3, "description": "Сравнить основные характеристики", "tool": null, "dependencies": [1,2]},
  {"step_number": 4, "description": "Сформулировать итоговый ответ", "tool": null, "dependencies": [3]}
]

Теперь сгенерируй план для задачи пользователя. Выдай только JSON.
"""
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=state["question"])]
    response = llm.invoke(messages)
    plan = parse_plan(response.content)
    print(f"✅ Сгенерирован план из {len(plan)} шагов.")
    for step in plan:
        print(f"   Шаг {step['step_number']}: {step['description']} (tool: {step.get('tool', 'нет')})")
    
    return {
        "plan": plan,
        "current_step": 0,
        "step_results": [],
        "plan_status": "active",
        "replan_count": 0
    }

def executor_node(state: PlanExecuteState) -> dict:
    """Выполняет текущий шаг плана."""
    plan = state["plan"]
    current_idx = state["current_step"]
    if current_idx >= len(plan):
        # Все шаги выполнены
        return {"plan_status": "completed"}
    
    step = plan[current_idx]
    print(f"\n⚙️  Выполнение шага {step['step_number']}: {step['description']}")
    
    # Проверяем зависимости: все ли предыдущие шаги выполнены успешно?
    deps = step.get("dependencies", [])
    step_results = state.get("step_results", [])
    for dep in deps:
        # Ищем результат для зависимого шага
        dep_result = next((r for r in step_results if r["step_number"] == dep), None)
        if not dep_result or not dep_result.get("success", False):
            # Зависимость не выполнена – пропускаем шаг или помечаем ошибку
            error_msg = f"Зависимость от шага {dep} не выполнена."
            print(f"❌ {error_msg}")
            # Сохраняем результат с ошибкой
            result_entry = {
                "step_number": step["step_number"],
                "result": error_msg,
                "success": False,
                "error": error_msg
            }
            return {
                "step_results": state.get("step_results", []) + [result_entry],
                "current_step": current_idx + 1,
                "plan_status": "failed"
            }
    
    # Выполняем шаг
    tool_name = step.get("tool")
    result_text = ""
    success = True
    error = None
    
    if tool_name and tool_name in tool_map:
        # Вызов инструмента
        try:
            # Для простоты передаём описание шага как запрос
            # В реальности нужно извлекать параметры из описания
            query = step["description"]
            tool_result = tool_map[tool_name].invoke({"query": query})
            result_text = str(tool_result)
            print(f"🔧 Инструмент {tool_name} вернул: {result_text[:100]}...")
        except Exception as e:
            error = str(e)
            success = False
            result_text = f"Ошибка при выполнении инструмента: {e}"
            print(f"❌ Ошибка инструмента: {e}")
    else:
        # Шаг без инструмента – используем LLM для генерации текста
        # Формируем контекст из предыдущих результатов
        context = ""
        for r in state.get("step_results", []):
            context += f"Результат шага {r['step_number']}: {r['result']}\n"
        
        prompt = f"""
Ты – исполнитель. Твоя задача – выполнить текущий шаг плана.

Текущий шаг: {step['description']}

Предыдущие результаты:
{context}

Выполни этот шаг. Если нужно сделать вывод или обобщение – сделай это. Ответ дай кратко и по делу.
"""
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            result_text = response.content
            print(f"📝 LLM сгенерировала: {result_text[:100]}...")
        except Exception as e:
            error = str(e)
            success = False
            result_text = f"Ошибка LLM: {e}"
            print(f"❌ Ошибка LLM: {e}")
    
    # Сохраняем результат
    result_entry = {
        "step_number": step["step_number"],
        "result": result_text,
        "success": success,
        "error": error
    }
    
    new_results = state.get("step_results", []) + [result_entry]
    new_idx = current_idx + 1
    
    # Если все шаги выполнены, меняем статус
    if new_idx >= len(plan):
        plan_status = "completed"
        print("✅ Все шаги выполнены.")
    else:
        plan_status = "active"
    
    return {
        "step_results": new_results,
        "current_step": new_idx,
        "plan_status": plan_status
    }

def replanner_node(state: PlanExecuteState) -> dict:
    """
    Узел перепланирования. Вызывается, если текущий шаг завершился с ошибкой.
    В упрощённой версии мы просто добавляем новый уточняющий шаг.
    В реальном проекте здесь можно использовать LLM для анализа и коррекции плана.
    """
    print("🔄 Перепланирование...")
    plan = state["plan"]
    current_idx = state["current_step"]
    
    # Находим последний неудачный шаг
    last_result = state["step_results"][-1] if state["step_results"] else None
    if not last_result or last_result.get("success", False):
        # Если ошибки нет, не перепланируем
        return {"plan_status": "active"}
    
    # Добавляем новый шаг для уточнения информации
    new_step = {
        "step_number": len(plan) + 1,
        "description": f"Уточнить информацию по шагу {last_result['step_number']} через web_search",
        "tool": "web_search",
        "dependencies": [last_result["step_number"]]
    }
    plan.append(new_step)
    print(f"➕ Добавлен новый шаг {new_step['step_number']}: {new_step['description']}")
    
    # Возвращаемся к выполнению нового шага
    return {
        "plan": plan,
        "plan_status": "active",
        "replan_count": state.get("replan_count", 0) + 1
    }

# ============================================================================
# 6. МАРШРУТИЗАЦИЯ
# ============================================================================

def route_after_planner(state: PlanExecuteState) -> Literal["executor", "finish"]:
    """После генерации плана переходим к выполнению."""
    if state.get("plan"):
        return "executor"
    else:
        return "finish"

def route_after_executor(state: PlanExecuteState) -> Literal["executor", "replanner", "finish"]:
    """Определяем, что делать после выполнения шага."""
    if state["plan_status"] == "completed":
        return "finish"
    elif state["plan_status"] == "failed":
        # Если ошибка и не превышен лимит перепланирований
        if state.get("replan_count", 0) < 3:
            return "replanner"
        else:
            print("⚠️ Превышен лимит перепланирований, завершаем.")
            return "finish"
    else:
        # Продолжаем выполнение следующего шага
        return "executor"

# ============================================================================
# 7. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(PlanExecuteState)
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)
builder.add_node("replanner", replanner_node)

builder.set_entry_point("planner")
builder.add_conditional_edges("planner", route_after_planner, {
    "executor": "executor",
    "finish": END
})
builder.add_conditional_edges("executor", route_after_executor, {
    "executor": "executor",
    "replanner": "replanner",
    "finish": END
})
builder.add_edge("replanner", "planner")  # после перепланирования снова в планировщик

graph = builder.compile()

# ============================================================================
# 8. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    question = "Сравни RAG и обычный ChatGPT. Опиши их основные различия и области применения."
    
    print("=" * 60)
    print(f"📝 Задача: {question}")
    print("=" * 60)
    
    initial_state = {
        "messages": [HumanMessage(content=question)],
        "question": question,
        "plan": [],
        "current_step": 0,
        "step_results": [],
        "plan_status": "active",
        "iteration": 0,
        "max_iterations": 10,
        "replan_count": 0
    }
    
    try:
        result = graph.invoke(initial_state, config={"recursion_limit": 20})
    except Exception as e:
        print(f"❌ Ошибка выполнения: {e}")
        exit(1)
    
    print("\n" + "=" * 60)
    print("📊 Итоговый план и результаты:")
    for step in result.get("plan", []):
        print(f"  Шаг {step['step_number']}: {step['description']}")
        # Найдём результат для этого шага
        step_res = next((r for r in result.get("step_results", []) if r["step_number"] == step["step_number"]), None)
        if step_res:
            status = "✅" if step_res.get("success") else "❌"
            print(f"    {status} Результат: {step_res['result'][:100]}...")
        else:
            print("    ⏳ Не выполнено")
    
    print(f"\n🏁 Статус: {result.get('plan_status')}")
    print(f"🔄 Перепланирований: {result.get('replan_count', 0)}")
```

---

### 3.7. Результаты тестирования

При запуске на задаче «Сравни RAG и ChatGPT» агент:

1. Сгенерировал план из 4 шагов (поиск RAG, поиск ChatGPT, сравнение, вывод).
2. Последовательно выполнил каждый шаг.
3. На шаге сравнения (без инструмента) использовал LLM для синтеза текста на основе предыдущих результатов.
4. Успешно завершил выполнение.

В случае ошибки (например, если инструмент вернул пустой результат) агент добавляет дополнительный уточняющий шаг и продолжает выполнение.

**Преимущества перед ReAct:**
- План виден целиком – пользователь может оценить его адекватность.
- Агент не теряет контекст, так как все результаты хранятся структурированно.
- Проще отлаживать – можно посмотреть, на каком шаге произошла ошибка.

---

### Краткий итог Тема 3

- **Plan‑and‑Execute** – паттерн для длительных задач, где сначала составляется полный план, а затем он выполняется.
- Основные узлы: **planner**, **executor**, **replanner**.
- План – это JSON-массив с шагами, зависимостями и указанием инструментов.
- При ошибке вызывается перепланировщик, который корректирует план (в нашей упрощённой версии – добавляет уточняющий шаг).
- Такой подход даёт **прозрачность**, **возможность параллельного выполнения** (в перспективе) и **устойчивость к ошибкам**.

---

**В следующей теме мы добавим самооценку (Reflexion) – научим агента критиковать свои ответы и улучшать их без внешнего вмешательства.**

## Тема 4. Самооценка и рефлексия (Reflexion)

В предыдущих темах мы научили агента планировать (Plan‑and‑Execute) и выполнять шаги. Но даже самый продуманный план может привести к некачественному результату – например, если поиск выдал нерелевантные данные, или LLM поверхностно проанализировала информацию. В нашем тесте Plan‑and‑Execute начиная с шага 4 модель повторяла один и тот же текст, не давая нового содержания. Проблема в том, что агент **не проверяет себя**. Он выполняет шаги механически, не задавая вопросов: «Достиг ли я цели?», «Полный ли ответ?», «Нет ли ошибок?». Именно здесь на помощь приходит паттерн **Reflexion (самооценка и рефлексия)**.

---

### 4.1. Концепция: учим агента критиковать себя

Reflexion добавляет в цикл работы агента **этап самокритики**. После выполнения шага (или всей задачи) агент:

1. **Оценивает** свой результат по заранее заданным критериям (корректность, полнота, соответствие цели).
2. **Формулирует обратную связь** – что сделано хорошо, что плохо, что можно улучшить.
3. Если оценка неудовлетворительная, агент **рефлексирует** – анализирует ошибки и генерирует **улучшенный план** или **корректирует уже выполненное действие**.
4. Повторяет выполнение с учётом исправлений, пока оценка не станет приемлемой или не будет превышено допустимое число попыток.

Этот подход имитирует поведение человека, который, написав текст, перечитывает его, исправляет ошибки и дорабатывает.

**Когда применять Reflexion:**
- Когда ответ должен быть **точным и полным** (аналитические отчёты, резюме).
- Когда агент работает с **неструктурированной информацией** и может допустить логические ошибки.
- В задачах, где **качество важнее скорости** (можно потратить несколько попыток на улучшение).

---

### 4.2. Узел `evaluator` – оцениваем результат

Оценщик (evaluator) – это LLM, которой мы передаём:
- исходную цель пользователя,
- результат, который нужно оценить (текст ответа или результат шага),
- критерии оценки (например, точность, полнота, ясность).

**Формат ответа оценщика** – структурированный JSON с полями:
- `score` – число от 1 до 10,
- `feedback` – текстовый комментарий с пояснением,
- `is_satisfactory` – логический флаг (достаточно ли хорош результат).

Для парсинга JSON мы используем **PydanticOutputParser** – это удобный способ задать схему ответа и гарантировать, что LLM выдаст валидный JSON.

**Пример промпта для оценщика** (сбалансированный, требует конкретики, но не штрафует за отсутствие цифр, если их нет в данных):

```
Ты – строгий, но справедливый критик. Оцени ответ по следующим критериям (каждый от 1 до 10):
1. Полнота: охвачены ли все аспекты вопроса?
2. Точность: нет ли фактических ошибок?
3. Ясность: легко ли понять ответ?
4. Наличие примеров: есть ли конкретные примеры использования, области применения?
5. Структурированность: хорошо ли организован ответ (пункты, разделы)?

Итоговая оценка – среднее арифметическое (округляй до целого).

Исходная цель: {question}
Результат агента: {answer}

{format_instructions}

Примечание: is_satisfactory = true, только если итоговая оценка >= 9.
Если примеров нет, но они объективно не могут быть получены из данных, снижай оценку не более чем на 1 балл.
```

**Код узла `evaluator_node`** (с использованием PydanticOutputParser):

```python
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class EvaluationResult(BaseModel):
    score: int = Field(description="Оценка от 1 до 10 (10 – идеально)")
    feedback: str = Field(description="Развёрнутая обратная связь с конкретными рекомендациями")
    is_satisfactory: bool = Field(description="true, если score >= 9")

def evaluator_node(state: PlanExecuteState) -> dict:
    print("\n🔍 Оценка результата...")
    final_answer = state.get("final_answer")
    if not final_answer:
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage):
                final_answer = msg.content
                break
        if not final_answer and state["step_results"]:
            final_answer = state["step_results"][-1].get("result", "Ответ не сгенерирован")
        else:
            final_answer = "Ответ не сгенерирован"

    parser = PydanticOutputParser(pydantic_object=EvaluationResult)
    format_instructions = parser.get_format_instructions()

    prompt = f"""
Ты – строгий, но справедливый критик. Оцени ответ по следующим критериям (каждый от 1 до 10):
1. Полнота: охвачены ли все аспекты вопроса?
2. Точность: нет ли фактических ошибок?
3. Ясность: легко ли понять ответ?
4. Наличие примеров: есть ли конкретные примеры использования, области применения?
5. Структурированность: хорошо ли организован ответ (пункты, разделы)?

Итоговая оценка – среднее арифметическое (округляй до целого).

Исходная цель: {state["question"]}
Результат агента: {final_answer}

{format_instructions}

Примечание: is_satisfactory = true, только если итоговая оценка >= 9.
Если примеров нет, но они объективно не могут быть получены из данных, снижай оценку не более чем на 1 балл.
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    try:
        eval_result = parser.parse(response.content)
        eval_result.is_satisfactory = eval_result.score >= 9
        print(f"📊 Оценка: {eval_result.score}/10")
        print(f"💬 Отзыв: {eval_result.feedback}")
        print(f"✅ Удовлетворительно: {eval_result.is_satisfactory}")
    except Exception as e:
        print(f"Ошибка парсинга оценки: {e}. Ставим 5/10.")
        eval_result = EvaluationResult(score=5, feedback="Не удалось распарсить оценку", is_satisfactory=False)

    return {
        "final_answer": final_answer,
        "retry_count": state.get("retry_count", 0),
        "eval_result": eval_result
    }
```

---

### 4.3. Узел `reflector` – рефлексия и исправление

Если оценка низкая (score < 9), вызывается **рефлектор** (reflector). Его задача – проанализировать ошибки и предложить **исправленный план** или **скорректированный ответ**.

В нашей реализации рефлектор принимает решение: если в обратной связи указано, что не хватает данных («не хватает», «дополнительная информация», «конкретные примеры»), он **добавляет новый шаг поиска** в план и возвращает управление исполнителю. В противном случае он генерирует **новый финальный ответ** с учётом критики.

**Пример промпта для рефлектора** (когда нужно просто улучшить ответ):

```
Ты – рефлектор. Твоя задача – улучшить ответ, учитывая критику.

Исходная цель: {question}
Предыдущий ответ: {answer}
Обратная связь оценщика: {feedback}

Улучши ответ. Обязательно:
- Если не хватает примеров – добавь конкретные области применения с пояснениями.
- Если ответ неструктурирован – сделай его чётким, используй пункты или таблицу.
- Исправь все указанные недостатки.

Дай новый, более качественный ответ.
```

**Код узла `reflector_node`** (с возможностью добавить поиск):

```python
def reflector_node(state: PlanExecuteState) -> dict:
    print("\n🤔 Рефлексия – улучшаем ответ...")
    retry_count = state.get("retry_count", 0) + 1

    eval_result = state.get("eval_result")
    if not eval_result:
        return {"retry_count": retry_count}

    history = state.get("reflection_history", [])
    history.append(f"Попытка {retry_count}: Оценка {eval_result.score}/10. Отзыв: {eval_result.feedback}")

    feedback_lower = eval_result.feedback.lower()
    need_more_data = any(phrase in feedback_lower for phrase in
                         ["не хватает", "дополнительная информация", "больше данных", "уточнить", "конкретные примеры"])
    if retry_count > 1 and eval_result.score <= 7:
        need_more_data = True

    if need_more_data and state.get("replan_count", 0) < 3:
        print("🔍 Рефлектор решил добавить новый шаг поиска для сбора данных.")
        plan = state.get("plan", [])
        new_step = {
            "step_number": len(plan) + 1,
            "description": "Поискать конкретные примеры и цифры по теме вопроса",
            "tool": "web_search",
            "dependencies": [len(plan)]
        }
        plan.append(new_step)
        print(f"➕ Добавлен новый шаг {new_step['step_number']}: {new_step['description']}")
        return {
            "plan": plan,
            "current_step": len(plan) - 1,
            "retry_count": retry_count,
            "reflection_history": history,
            "plan_status": "active",
            "reflector_added_step": True,
            "replan_count": state.get("replan_count", 0) + 1
        }
    else:
        prompt = f"""
Ты – рефлектор. Твоя задача – улучшить ответ, учитывая критику.

Исходная цель: {state["question"]}
Предыдущий ответ: {state["final_answer"]}
Обратная связь оценщика: {eval_result.feedback}

Улучши ответ. Обязательно:
- Если не хватает примеров – добавь конкретные области применения с пояснениями.
- Если ответ неструктурирован – сделай его чётким, используй пункты или таблицу.
- Исправь все указанные недостатки.

Дай новый, более качественный ответ.
"""
        response = llm.invoke([HumanMessage(content=prompt)])
        new_answer = response.content.strip()
        print(f"📝 Сгенерирован улучшенный ответ (попытка {retry_count})")
        return {
            "final_answer": new_answer,
            "retry_count": retry_count,
            "reflection_history": history,
            "messages": [AIMessage(content=new_answer)],
            "reflector_added_step": False
        }
```

---

### 4.4. Интеграция в граф

В графе после завершения всех шагов (статус `completed`) вместо немедленного завершения мы направляем поток в **`evaluator`**. Затем:

- Если оценка удовлетворительная (`is_satisfactory == True`) – переходим в `END`.
- Если нет и количество попыток (`retry_count`) меньше `max_retries` – переходим в **`reflector`**.
- После `reflector` проверяем флаг `reflector_added_step`:
  - Если `True` (был добавлен новый шаг) – возвращаемся в **`executor`** для выполнения нового шага.
  - Иначе – снова в **`evaluator`** для переоценки улучшенного ответа.
- Если попытки исчерпаны – завершаем с последним ответом.

**Фрагмент маршрутизации:**

```python
def route_after_executor(state: PlanExecuteState) -> Literal["executor", "replanner", "evaluator", "finish"]:
    if state["plan_status"] == "completed":
        return "evaluator"
    elif state["plan_status"] == "failed":
        return "replanner" if state.get("replan_count", 0) < 3 else "finish"
    else:
        return "executor"

def route_after_evaluator(state: PlanExecuteState) -> Literal["reflector", "finish"]:
    eval_result = state.get("eval_result")
    retry_count = state.get("retry_count", 0)
    max_retries = state.get("max_retries", 4)
    if eval_result and eval_result.is_satisfactory:
        return "finish"
    elif retry_count < max_retries:
        return "reflector"
    else:
        return "finish"

def route_after_reflector(state: PlanExecuteState) -> Literal["executor", "evaluator", "finish"]:
    if state.get("reflector_added_step", False) and state.get("plan_status") == "active":
        return "executor"
    else:
        return "evaluator"
```

**Добавление узлов и рёбер в граф:**

```python
builder = StateGraph(PlanExecuteState)
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)
builder.add_node("replanner", replanner_node)
builder.add_node("evaluator", evaluator_node)
builder.add_node("reflector", reflector_node)

builder.set_entry_point("planner")
builder.add_conditional_edges("planner", route_after_planner, {...})
builder.add_conditional_edges("executor", route_after_executor, {...})
builder.add_edge("replanner", "planner")
builder.add_conditional_edges("evaluator", route_after_evaluator, {...})
builder.add_conditional_edges("reflector", route_after_reflector, {...})
```

---

### 4.5. Ограничение числа попыток (retries)

Чтобы избежать бесконечных циклов, мы вводим счётчик `retry_count` и максимальное число попыток `max_retries` (в примере – 4). Если после четырёх итераций оценка всё ещё низкая, агент завершает работу и выдаёт последний сгенерированный ответ.

**В состоянии** добавляем поля:
- `retry_count: int` – текущее число попыток рефлексии.
- `max_retries: int` – максимальное допустимое число попыток.
- `eval_result: Optional[EvaluationResult]` – последняя оценка (используется в рефлекторе и маршрутизации).
- `reflector_added_step: bool` – флаг, указывающий, добавил ли рефлектор новый шаг в план (чтобы после выполнения нового шага снова пойти в оценщик).

**Пример инициализации состояния:**

```python
initial_state = {
    ...
    "retry_count": 0,
    "max_retries": 4,
    "eval_result": None,
    "reflection_history": [],
    "reflector_added_step": False
}
```

---

### 4.6. Полный код агента

Все описанные выше узлы – `planner`, `executor`, `replanner`, `evaluator`, `reflector` – объединены в единый скрипт, который реализует полный цикл работы агента: планирование → исполнение → самооценка → рефлексия (с возможностью добавления новых шагов) → повторная оценка, с защитой от зацикливания.

**Полный код агента представлен ниже `reflexion_agent_final.py` **  
В этом файле содержатся:
- Все необходимые импорты и настройка LLM (Ollama с моделью `qwen2.5:3b`).
- Инструменты: `search_docs`, `web_search`, `calculate`.
- Определение состояния `PlanExecuteState` и модели оценки `EvaluationResult`.
- Узлы графа: `planner_node`, `executor_node`, `replanner_node`, `evaluator_node`, `reflector_node`.
- Функции маршрутизации и сборка графа на основе `StateGraph`.
- Тестовый запуск на задаче сравнения RAG и ChatGPT с выводом деталей выполнения, истории рефлексии и итоговой оценки.

скопируйте код и запустите его в своём окружении, чтобы увидеть работу автономного агента с самооценкой в действии.



```python
"""
reflexion_agent_final.py - Полностью переработанный агент с Plan-and-Execute и Reflexion
Лекция 6.8 – Финальная версия

Цель: добиться оценки >= 9/10 за счёт улучшенных промптов,
динамического перепланирования при нехватке данных и структурированных ответов.

Ключевые улучшения:
- Промпты исполнителя адаптируются под каждый шаг, давая уникальные результаты.
- Оценщик сбалансирован: требует конкретики, но не штрафует за отсутствие цифр, если их нет в данных.
- Рефлектор может инициировать дополнительный поиск, если в ответе не хватает фактов.
- Добавлена постобработка ответов для улучшения читаемости.
- Более детальное логирование и история рефлексии.
"""

import json
import re
from typing import TypedDict, List, Annotated, Literal, Optional, Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# ============================================================================
# 1. ИНСТРУМЕНТЫ
# ============================================================================

@tool
def search_docs(query: str) -> str:
    """Ищет информацию в локальной базе знаний (заглушка)."""
    if "RAG" in query.upper():
        return ("RAG (Retrieval-Augmented Generation) — подход, который комбинирует поиск по внешней базе знаний "
                "и генерацию текста. Это позволяет моделям отвечать точнее и актуальнее.")
    elif "LLM" in query.upper():
        return "Большие языковые модели (LLM) обучаются на огромных текстовых корпусах и генерируют текст."
    else:
        return "Информация не найдена."

# Реальный веб-поиск
web_search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Выполняет поиск в интернете через DuckDuckGo (без API-ключа)."""
    try:
        results = web_search_tool.invoke(query)
        # Ограничиваем длину, чтобы не перегружать контекст
        return results[:1500] if len(results) > 1500 else results
    except Exception as e:
        return f"Ошибка поиска: {e}"

@tool
def calculate(expression: str) -> str:
    """Безопасное выполнение математических выражений."""
    if not re.match(r'^[\d+\-*/().\s]+$', expression):
        return "Ошибка: недопустимые символы."
    try:
        result = eval(expression, {"__builtins__": {}})
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

tools = [search_docs, web_search, calculate]
tool_map = {t.name: t for t in tools}

# ============================================================================
# 2. НАСТРОЙКА LLM
# ============================================================================

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=1024)  # увеличили для длинных ответов

# ============================================================================
# 3. МОДЕЛЬ ДЛЯ ОЦЕНКИ (Pydantic)
# ============================================================================

class EvaluationResult(BaseModel):
    score: int = Field(description="Оценка от 1 до 10 (10 – идеально)")
    feedback: str = Field(description="Развёрнутая обратная связь с конкретными рекомендациями")
    is_satisfactory: bool = Field(description="true, если score >= 9")

# ============================================================================
# 4. СОСТОЯНИЕ
# ============================================================================

class PlanExecuteState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    plan: List[Dict[str, Any]]           # план в виде списка шагов
    current_step: int                    # индекс текущего шага
    step_results: List[Dict[str, Any]]   # результаты каждого шага
    plan_status: Literal["active", "completed", "failed"]
    iteration: int
    max_iterations: int
    replan_count: int                    # счётчик перепланирований
    final_answer: Optional[str]          # итоговый ответ для оценки
    retry_count: int                     # число попыток рефлексии
    max_retries: int                     # максимум попыток
    eval_result: Optional[EvaluationResult]
    reflection_history: List[str]        # история рефлексии (для контекста)
    # Флаг, что рефлектор добавил новый шаг в план
    reflector_added_step: bool

# ============================================================================
# 5. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================================

def parse_plan(response_content: str) -> List[Dict[str, Any]]:
    """Извлекает JSON-план из ответа LLM."""
    try:
        json_match = re.search(r'\[.*\]', response_content, re.DOTALL)
        if json_match:
            plan = json.loads(json_match.group())
            if isinstance(plan, list) and all(isinstance(item, dict) for item in plan):
                # Убедимся, что все шаги имеют поля
                for step in plan:
                    step.setdefault("tool", None)
                    step.setdefault("dependencies", [])
                return plan
    except Exception as e:
        print(f"Ошибка парсинга плана: {e}")
    # План по умолчанию
    return [{"step_number": 1, "description": "Ответить на вопрос", "tool": None, "dependencies": []}]

def format_step_result(step_num: int, result: str) -> str:
    """Форматирует результат шага для контекста."""
    return f"Шаг {step_num}: {result}"

# ============================================================================
# 6. УЗЛЫ ГРАФА
# ============================================================================

def planner_node(state: PlanExecuteState) -> dict:
    """Генерирует начальный план или перепланирует, если была ошибка."""
    print("\n📋 Генерация плана...")

    # Если уже есть план и replan_count > 0, значит перепланируем.
    if state.get("plan") and state.get("replan_count", 0) > 0:
        # Добавляем шаг для уточнения информации
        plan = state["plan"]
        new_step = {
            "step_number": len(plan) + 1,
            "description": "Поискать дополнительную информацию через web_search, чтобы уточнить детали",
            "tool": "web_search",
            "dependencies": [state["current_step"]]
        }
        plan.append(new_step)
        print(f"🔄 Перепланирование: добавлен шаг {new_step['step_number']} – {new_step['description']}")
        return {"plan": plan, "replan_count": state.get("replan_count", 0) + 1}

    # Генерация нового плана – улучшенный промпт
    system_prompt = """
Ты – планировщик. Разбей цель пользователя на чёткую последовательность шагов.
Каждый шаг должен быть конкретным и, если требуется, использовать инструмент.
Доступные инструменты: search_docs, web_search, calculate.
Если инструмент не нужен, укажи "tool": null.

Формат ответа – JSON-массив, где каждый элемент:
{
  "step_number": номер (с 1),
  "description": краткое описание действия,
  "tool": имя инструмента или null,
  "dependencies": [номера шагов, от которых зависит]
}

Старайся, чтобы шаги без инструментов были разными: сравнение, анализ, формулировка выводов.
Пример для "Сравни RAG и ChatGPT":
[
  {"step_number":1,"description":"Найти информацию о RAG","tool":"web_search","dependencies":[]},
  {"step_number":2,"description":"Найти информацию о ChatGPT","tool":"web_search","dependencies":[]},
  {"step_number":3,"description":"Сравнить архитектуру и подходы","tool":null,"dependencies":[1,2]},
  {"step_number":4,"description":"Сравнить области применения с примерами","tool":null,"dependencies":[1,2]},
  {"step_number":5,"description":"Сформулировать итоговый вывод","tool":null,"dependencies":[3,4]}
]

Теперь сгенерируй план для задачи пользователя. Только JSON.
"""
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=state["question"])]
    response = llm.invoke(messages)
    plan = parse_plan(response.content)
    print(f"✅ Сгенерирован план из {len(plan)} шагов.")
    for step in plan:
        print(f"   Шаг {step['step_number']}: {step['description']} (tool: {step.get('tool', 'нет')})")

    return {
        "plan": plan,
        "current_step": 0,
        "step_results": [],
        "plan_status": "active",
        "replan_count": 0,
        "reflector_added_step": False
    }

def executor_node(state: PlanExecuteState) -> dict:
    """Выполняет текущий шаг с контекстно-зависимым промптом."""
    plan = state["plan"]
    current_idx = state["current_step"]
    if current_idx >= len(plan):
        return {"plan_status": "completed"}

    step = plan[current_idx]
    print(f"\n⚙️  Выполнение шага {step['step_number']}: {step['description']}")

    # Проверка зависимостей
    deps = step.get("dependencies", [])
    step_results = state.get("step_results", [])
    for dep in deps:
        dep_result = next((r for r in step_results if r["step_number"] == dep), None)
        if not dep_result or not dep_result.get("success", False):
            error_msg = f"Зависимость от шага {dep} не выполнена."
            print(f"❌ {error_msg}")
            result_entry = {
                "step_number": step["step_number"],
                "result": error_msg,
                "success": False,
                "error": error_msg
            }
            return {
                "step_results": state.get("step_results", []) + [result_entry],
                "current_step": current_idx + 1,
                "plan_status": "failed"
            }

    # Выполнение
    tool_name = step.get("tool")
    result_text = ""
    success = True
    error = None

    if tool_name and tool_name in tool_map:
        try:
            query = step["description"]
            tool_result = tool_map[tool_name].invoke({"query": query})
            result_text = str(tool_result)
            # Обрезаем слишком длинные результаты
            if len(result_text) > 1000:
                result_text = result_text[:1000] + "... (обрезано)"
            print(f"🔧 Инструмент {tool_name} вернул: {result_text[:100]}...")
        except Exception as e:
            error = str(e)
            success = False
            result_text = f"Ошибка инструмента: {e}"
            print(f"❌ Ошибка: {e}")
    else:
        # Шаг без инструмента – используем LLM с улучшенным промптом
        # Собираем контекст из предыдущих шагов
        context = ""
        for r in state.get("step_results", []):
            context += format_step_result(r["step_number"], r["result"]) + "\n"

        # Определяем тип шага по описанию для точной настройки промпта
        desc_lower = step["description"].lower()
        if "сравни" in desc_lower or "сравнить" in desc_lower or "отличия" in desc_lower:
            task_type = "comparison"
        elif "примен" in desc_lower or "использование" in desc_lower or "области" in desc_lower:
            task_type = "application"
        elif "вывод" in desc_lower or "итог" in desc_lower or "резюми" in desc_lower:
            task_type = "summary"
        else:
            task_type = "general"

        # Подбираем инструкцию в зависимости от типа
        if task_type == "comparison":
            instruction = "Выдели ключевые различия в виде пунктов. Для каждого пункта укажи, чем отличается RAG от ChatGPT. Старайся дать конкретные характеристики."
        elif task_type == "application":
            instruction = "Перечисли минимум 3 конкретные области применения. Для каждой области кратко поясни, почему эта технология подходит."
        elif task_type == "summary":
            instruction = "Сформулируй чёткий итоговый вывод, обобщи всё, что было сказано ранее. Дай рекомендацию, что лучше использовать в каких случаях."
        else:
            instruction = "Дай развёрнутый ответ, не повторяя предыдущие выводы."

        prompt = f"""
Ты – исполнитель, выполняешь конкретный шаг плана.

Текущий шаг: {step['description']}

Предыдущие результаты:
{context}

Задача: {instruction}

Ответь кратко, но содержательно. Если нужно, используй структуру (пункты, список).
"""
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            result_text = response.content.strip()
            print(f"📝 LLM сгенерировала: {result_text[:100]}...")
        except Exception as e:
            error = str(e)
            success = False
            result_text = f"Ошибка LLM: {e}"
            print(f"❌ Ошибка: {e}")

    result_entry = {
        "step_number": step["step_number"],
        "result": result_text,
        "success": success,
        "error": error
    }
    new_results = state.get("step_results", []) + [result_entry]
    new_idx = current_idx + 1

    if new_idx >= len(plan):
        plan_status = "completed"
        print("✅ Все шаги выполнены.")
    else:
        plan_status = "active"

    return {
        "step_results": new_results,
        "current_step": new_idx,
        "plan_status": plan_status
    }

def replanner_node(state: PlanExecuteState) -> dict:
    """Перепланирование при ошибке – добавляет поисковый шаг."""
    print("🔄 Перепланирование из-за ошибки...")
    plan = state["plan"]
    last_result = state["step_results"][-1] if state["step_results"] else None
    if not last_result or last_result.get("success", False):
        return {"plan_status": "active"}

    new_step = {
        "step_number": len(plan) + 1,
        "description": f"Найти дополнительную информацию по теме шага {last_result['step_number']}",
        "tool": "web_search",
        "dependencies": [last_result["step_number"]]
    }
    plan.append(new_step)
    print(f"➕ Добавлен шаг {new_step['step_number']}: {new_step['description']}")
    return {
        "plan": plan,
        "plan_status": "active",
        "replan_count": state.get("replan_count", 0) + 1,
        "reflector_added_step": False
    }

def evaluator_node(state: PlanExecuteState) -> dict:
    """
    Оценивает финальный ответ по сбалансированным критериям.
    Критерии: полнота, точность, ясность, наличие примеров, структурированность.
    Порог для удовлетворительности – 9 баллов.
    """
    print("\n🔍 Оценка результата...")

    final_answer = state.get("final_answer")
    if not final_answer:
        # Попытка извлечь последний ответ из сообщений
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage):
                final_answer = msg.content
                break
        if not final_answer and state["step_results"]:
            final_answer = state["step_results"][-1].get("result", "Ответ не сгенерирован")
        else:
            final_answer = "Ответ не сгенерирован"

    parser = PydanticOutputParser(pydantic_object=EvaluationResult)
    format_instructions = parser.get_format_instructions()

    prompt = f"""
Ты – строгий, но справедливый критик. Оцени ответ по следующим критериям (каждый от 1 до 10):
1. Полнота: охвачены ли все аспекты вопроса?
2. Точность: нет ли фактических ошибок?
3. Ясность: легко ли понять ответ?
4. Наличие примеров: есть ли конкретные примеры использования, области применения?
5. Структурированность: хорошо ли организован ответ (пункты, разделы)?

Итоговая оценка – среднее арифметическое (округляй до целого).

Исходная цель: {state["question"]}
Результат агента: {final_answer}

{format_instructions}

Примечание: is_satisfactory = true, только если итоговая оценка >= 9.
Если примеров нет, но они объективно не могут быть получены из данных, снижай оценку не более чем на 1 балл.
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    try:
        eval_result = parser.parse(response.content)
        # Гарантируем, что is_satisfactory соответствует порогу
        eval_result.is_satisfactory = eval_result.score >= 9
        print(f"📊 Оценка: {eval_result.score}/10")
        print(f"💬 Отзыв: {eval_result.feedback}")
        print(f"✅ Удовлетворительно: {eval_result.is_satisfactory}")
    except Exception as e:
        print(f"Ошибка парсинга оценки: {e}. Ставим 5/10.")
        eval_result = EvaluationResult(score=5, feedback="Не удалось распарсить оценку", is_satisfactory=False)

    return {
        "final_answer": final_answer,
        "retry_count": state.get("retry_count", 0),
        "eval_result": eval_result
    }

def reflector_node(state: PlanExecuteState) -> dict:
    """
    Рефлектор – анализирует обратную связь и решает, как улучшить ответ.
    Если не хватает данных, инициирует новый поиск (добавляет шаг в план).
    Иначе генерирует улучшенный финальный ответ.
    """
    print("\n🤔 Рефлексия – улучшаем ответ...")
    retry_count = state.get("retry_count", 0) + 1

    eval_result = state.get("eval_result")
    if not eval_result:
        return {"retry_count": retry_count}

    # Сохраняем историю
    history = state.get("reflection_history", [])
    history.append(f"Попытка {retry_count}: Оценка {eval_result.score}/10. Отзыв: {eval_result.feedback}")

    # Проверяем, нужно ли добавить поиск.
    # Если в фидбеке есть слова "не хватает", "дополнительная информация", "больше данных", то добавим поиск.
    feedback_lower = eval_result.feedback.lower()
    need_more_data = any(phrase in feedback_lower for phrase in
                         ["не хватает", "дополнительная информация", "больше данных", "уточнить", "конкретные примеры"])

    # Если уже были попытки рефлексии, и оценка не растёт, тоже можно попробовать поиск.
    if retry_count > 1 and eval_result.score <= 7:
        need_more_data = True

    if need_more_data and state.get("replan_count", 0) < 3:
        # Добавляем новый шаг поиска в план
        print("🔍 Рефлектор решил добавить новый шаг поиска для сбора данных.")
        plan = state.get("plan", [])
        new_step = {
            "step_number": len(plan) + 1,
            "description": "Поискать конкретные примеры и цифры по теме вопроса",
            "tool": "web_search",
            "dependencies": [len(plan)]  # зависит от последнего шага
        }
        plan.append(new_step)
        print(f"➕ Добавлен новый шаг {new_step['step_number']}: {new_step['description']}")
        # Возвращаем обновлённый план, сбрасываем current_step на новый шаг
        return {
            "plan": plan,
            "current_step": len(plan) - 1,  # перейдём к новому шагу
            "retry_count": retry_count,
            "reflection_history": history,
            "plan_status": "active",
            "reflector_added_step": True,
            "replan_count": state.get("replan_count", 0) + 1
        }
    else:
        # Иначе просто генерируем улучшенный финальный ответ
        prompt = f"""
Ты – рефлектор. Твоя задача – улучшить ответ, учитывая критику.

Исходная цель: {state["question"]}
Предыдущий ответ: {state["final_answer"]}
Обратная связь оценщика: {eval_result.feedback}

Улучши ответ. Обязательно:
- Если не хватает примеров – добавь конкретные области применения с пояснениями.
- Если ответ неструктурирован – сделай его чётким, используй пункты или таблицу.
- Исправь все указанные недостатки.

Дай новый, более качественный ответ.
"""
        response = llm.invoke([HumanMessage(content=prompt)])
        new_answer = response.content.strip()
        print(f"📝 Сгенерирован улучшенный ответ (попытка {retry_count})")
        return {
            "final_answer": new_answer,
            "retry_count": retry_count,
            "reflection_history": history,
            "messages": [AIMessage(content=new_answer)],
            "reflector_added_step": False
        }

# ============================================================================
# 7. МАРШРУТИЗАЦИЯ
# ============================================================================

def route_after_planner(state: PlanExecuteState) -> Literal["executor", "finish"]:
    return "executor" if state.get("plan") else "finish"

def route_after_executor(state: PlanExecuteState) -> Literal["executor", "replanner", "evaluator", "finish"]:
    if state["plan_status"] == "completed":
        return "evaluator"
    elif state["plan_status"] == "failed":
        if state.get("replan_count", 0) < 3:
            return "replanner"
        else:
            return "finish"
    else:
        return "executor"

def route_after_evaluator(state: PlanExecuteState) -> Literal["reflector", "finish"]:
    eval_result = state.get("eval_result")
    retry_count = state.get("retry_count", 0)
    max_retries = state.get("max_retries", 4)  # увеличили до 4

    if eval_result and eval_result.is_satisfactory:
        print("✅ Оценка отличная, завершаем.")
        return "finish"
    elif retry_count < max_retries:
        print("🔄 Оценка низкая, запускаем рефлексию.")
        return "reflector"
    else:
        print("⚠️ Превышено число попыток, завершаем.")
        return "finish"

def route_after_reflector(state: PlanExecuteState) -> Literal["executor", "evaluator", "finish"]:
    """
    После рефлектора:
    - Если рефлектор добавил новый шаг в план – переходим к executor.
    - Иначе снова к evaluator для переоценки улучшенного ответа.
    """
    if state.get("reflector_added_step", False) and state.get("plan_status") == "active":
        return "executor"
    else:
        return "evaluator"

# ============================================================================
# 8. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(PlanExecuteState)
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)
builder.add_node("replanner", replanner_node)
builder.add_node("evaluator", evaluator_node)
builder.add_node("reflector", reflector_node)

builder.set_entry_point("planner")
builder.add_conditional_edges("planner", route_after_planner, {
    "executor": "executor",
    "finish": END
})
builder.add_conditional_edges("executor", route_after_executor, {
    "executor": "executor",
    "replanner": "replanner",
    "evaluator": "evaluator",
    "finish": END
})
builder.add_edge("replanner", "planner")
builder.add_conditional_edges("evaluator", route_after_evaluator, {
    "reflector": "reflector",
    "finish": END
})
builder.add_conditional_edges("reflector", route_after_reflector, {
    "executor": "executor",
    "evaluator": "evaluator",
    "finish": END
})

graph = builder.compile()

# ============================================================================
# 9. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    question = "Сравни RAG и обычный ChatGPT. Опиши их основные различия и области применения."

    print("=" * 60)
    print(f"📝 Задача: {question}")
    print("=" * 60)

    initial_state = {
        "messages": [HumanMessage(content=question)],
        "question": question,
        "plan": [],
        "current_step": 0,
        "step_results": [],
        "plan_status": "active",
        "iteration": 0,
        "max_iterations": 15,
        "replan_count": 0,
        "final_answer": None,
        "retry_count": 0,
        "max_retries": 4,
        "eval_result": None,
        "reflection_history": [],
        "reflector_added_step": False
    }

    try:
        result = graph.invoke(initial_state, config={"recursion_limit": 50})
    except Exception as e:
        print(f"❌ Ошибка выполнения: {e}")
        exit(1)

    print("\n" + "=" * 60)
    print("📊 ИТОГОВЫЙ ОТВЕТ:")
    final_answer = result.get("final_answer")
    if final_answer:
        print(final_answer)
    else:
        print("Ответ не сгенерирован")

    print(f"\n🔄 Попыток рефлексии: {result.get('retry_count', 0)}")
    eval_res = result.get("eval_result")
    if eval_res:
        print(f"📊 Итоговая оценка: {eval_res.score}/10")
        print(f"💬 Отзыв: {eval_res.feedback}")
    print(f"🏁 Статус плана: {result.get('plan_status')}")

    if result.get("reflection_history"):
        print("\n📜 История рефлексии:")
        for entry in result["reflection_history"]:
            print(f"  - {entry}")

    # Вывод всех шагов и их результатов для наглядности
    print("\n📋 Детали выполнения шагов:")
    for step in result.get("plan", []):
        step_res = next((r for r in result.get("step_results", []) if r["step_number"] == step["step_number"]), None)
        status = "✅" if step_res and step_res.get("success") else "❌" if step_res else "⏳"
        print(f"  Шаг {step['step_number']}: {step['description']} {status}")
        if step_res and step_res.get("result"):
            print(f"    Результат: {step_res['result'][:200]}...")
```

---

### Краткий итог Тема 4

- **Reflexion** – паттерн, который добавляет **самокритику** в работу агента.
- **Оценщик (evaluator)** – LLM, которая выставляет оценку и пишет обратную связь (структурированный JSON).
- **Рефлектор (reflector)** – LLM, которая улучшает ответ на основе обратной связи, при необходимости добавляя новые шаги в план.
- Цикл «оценка → рефлексия → (выполнение нового шага или повторная оценка)» продолжается, пока качество не станет удовлетворительным или не исчерпается лимит попыток.
- Это повышает **надёжность** и **качество** финального ответа, особенно в сложных, многошаговых задачах.

---

В следующей теме мы соберём все три паттерна (ReAct, Plan‑and‑Execute, Reflexion) в единого суперагента и протестируем его на реальных задачах.

## Финальная часть: Тема 5. Долгосрочная память и управление состоянием + Тема 6. Обработка ошибок и ограничения + Итоговый код

В предыдущих разделах мы построили агента, который планирует, выполняет, оценивает и рефлексирует. Однако в реальных приложениях агент должен работать надолго, запоминать свои действия, восстанавливаться после сбоев и корректно реагировать на ошибки. В финальной части мы добавим **долгосрочную память**, **управление состоянием**, **обработку ошибок** и **логирование**, чтобы сделать агента готовым к промышленному использованию. В итоге мы получим полностью автономного агента, способного справляться со сложными задачами, требующими множества шагов и самокоррекции.

---

### Тема 5. Долгосрочная память и управление состоянием

Наш агент работает в рамках одного сеанса: он получает вопрос, строит план, выполняет, оценивает и завершается. Но в реальных системах агент должен **помнить** свои предыдущие действия, чтобы не повторять их, а также уметь **восстанавливаться** после перезапусков. Для этого мы вводим долгосрочную память и эффективное управление состоянием.

#### 5.1. Хранение истории выполненных шагов, промежуточных выводов и планов

В наших предыдущих реализациях состояние (`PlanExecuteState`) содержало всю историю диалога, план, результаты шагов и т.д. Однако после завершения работы агента это состояние терялось. Чтобы сохранить его между запусками, мы используем **`MemorySaver`** – механизм LangGraph, который автоматически сохраняет состояние после каждого шага и позволяет восстановить его при следующем вызове.

В нашем финальном коде мы добавляем:

```python
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)
```

Теперь при вызове мы передаём `config` с уникальным `thread_id`:

```python
config = {"configurable": {"thread_id": "user_session_123"}}
result = graph.invoke(initial_state, config)
```

Благодаря этому состояние сохраняется между запусками, и агент может продолжать выполнение с того места, где остановился.

#### 5.2. Добавление краткого резюме предыдущих шагов в контекст

В длительных сессиях количество сообщений и результатов может стать слишком большим для контекстного окна LLM. Чтобы избежать перегрузки, мы храним **краткое резюме** (`summary`) выполненных шагов. В состоянии добавляется поле `summary: str`, которое обновляется после каждого выполненного шага:

```python
def update_summary(state: PlanExecuteState, step_result: dict) -> str:
    old_summary = state.get("summary", "")
    new_info = f"Шаг {step_result['step_number']}: {step_result['result'][:150]}..."
    if len(old_summary) + len(new_info) > 2000:
        old_summary = old_summary[-1500:]  # обрезаем, если слишком длинное
    return old_summary + "\n" + new_info
```

При формировании промпта для LLM (в `executor_node` и `reflector_node`) мы используем `state["summary"]` вместо полной истории `step_results`, что экономит токены и ускоряет обработку.

#### 5.3. Восстановление прерванного выполнения

`MemorySaver` хранит состояние в оперативной памяти, что не подходит для долгосрочного хранения. Для продакшена мы можем заменить его на `SqliteSaver` или `PostgresSaver` – они сохраняют состояние в файл или базу данных, позволяя восстановить выполнение даже после перезапуска сервера.

В нашем коде используется `MemorySaver`, но замена на `SqliteSaver` тривиальна:

```python
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string("checkpoints.db") as saver:
    graph = builder.compile(checkpointer=saver)
```

Таким образом, агент становится устойчивым к сбоям и может продолжать длительные задачи, начатые ранее.

---

### Тема 6. Обработка ошибок и ограничения

Даже самый продуманный агент может столкнуться с неожиданными ситуациями: зацикливанием, некорректными планами, неработающими инструментами. В нашем финальном коде мы реализовали несколько механизмов, которые делают агента **надёжным** и **безопасным**.

#### 6.1. Защита от зацикливания и таймауты

Мы используем глобальный счётчик `max_iterations` (установлен в 15) и счётчик попыток рефлексии `max_retries` (4). Кроме того, в `executor_node` добавлен таймаут на выполнение одного шага (30 секунд):

```python
start_time = time.time()
timeout = 30
# ... выполнение шага
if time.time() - start_time > timeout:
    success = False
    result_text = "Превышен таймаут выполнения шага"
```

Эти меры предотвращают бесконечные циклы и зависания.

#### 6.2. Некорректный план: fallback с human‑in‑the‑loop

Если планировщик генерирует невалидный план (например, содержит циклы или отсутствующие зависимости), мы запрашиваем уточнение у пользователя. Для этого в состояние добавлен флаг `need_user_input` и узел `human_input_node`, который выводит запрос и ожидает ввод.

После получения ответа пользователя мы обновляем `question` и сбрасываем план для перегенерации.

#### 6.3. Семантическое обнаружение тупика (стагнации)

Иногда агент не зацикливается в прямом смысле (итерации растут), но состояние не улучшается: оценка не растёт, данные не добавляются. Мы реализовали функцию `detect_stagnation()`, которая анализирует историю оценок и, если последние 2–3 оценки одинаковы или не растут, а также не было добавления новых данных, завершает работу с сообщением о тупике.

```python
def detect_stagnation(state: PlanExecuteState) -> bool:
    history = state.get("reflection_history", [])
    if len(history) < 2:
        return False
    scores = [...]
    if len(scores) >= 2:
        if len(set(scores)) == 1 and scores[0] <= 8:
            return True
        if len(scores) >= 2 and scores[-1] <= scores[-2] and len(history) >= 3:
            return True
    return False
```

Это позволяет агенту не тратить время впустую, когда улучшения невозможны.

#### 6.4. Логирование всех мыслей, действий и оценок

Для отладки и аудита мы используем стандартный модуль `logging`. В каждом узле добавляются информационные сообщения:

```python
import logging
logger = logging.getLogger("AutonomousAgent")
logger.info("Планировщик: сгенерирован план...")
logger.info("Исполнитель: шаг выполнен...")
logger.info("Оценщик: оценка ...")
```

Логи выводятся в консоль и могут быть сохранены в файл для последующего анализа.

#### 6.5. Практические советы

- **Начинайте с малого**: сначала тестируйте на задачах с 2–3 шагами, затем усложняйте.
- **Пробуйте разные модели**: `qwen2.5:3b` хорошо для демонстрации, но для продакшена используйте более мощные модели (например, `llama3.1`, `gpt-4o`).
- **Следите за токенами**: используйте резюмирование (`summary`) и обрезайте длинные результаты поиска.
- **Тестируйте стагнацию**: если агент перестаёт улучшаться, проверьте, не требуется ли смена стратегии или добавление новых инструментов.

---

### Итоговый код: полностью автономный агент

Ниже представлен **финальный код** нашего агента, который объединяет все изученные паттерны и улучшения:

- **Планирование** (Plan‑and‑Execute) с генерацией плана в JSON.
- **Исполнение** с использованием инструментов (`web_search`, `search_docs`, `calculate`).
- **Самооценка** (evaluator) с выставлением баллов и обратной связью.
- **Рефлексия** (reflector) – улучшение ответа или добавление новых шагов для сбора недостающих данных.
- **Агрегация данных** – сбор фактов, цифр и примеров из найденных источников.
- **Факт-чекер** – проверка, что цифры и примеры взяты из реальных данных.
- **Долгосрочная память** – сохранение состояния через `MemorySaver`.
- **Обработка ошибок** – таймауты, детекция стагнации, human‑in‑the‑loop.
- **Логирование** – полный трейс выполнения.

**Сохраните код как `autonomous_agent.py` и запустите его** – агент самостоятельно выполнит задачу сравнения RAG и ChatGPT, соберёт конкретные примеры и цифры, оценит свой ответ и, при необходимости, улучшит его, используя рефлексию и дополнительные поиски.

```python
"""
autonomous_agent.py - Автономный агент (финальная версия, улучшенная до 9/10)
Лекция 6.8 – Итоговый код

Улучшения:
- Промпты оптимизированы для получения конкретных примеров и цифр из источников
- Добавлен агрегатор для сбора и структурирования данных из поиска
- Добавлен факт-чекер, проверяющий, что цифры и примеры взяты из найденных источников
- Рефлектор генерирует конкретные поисковые запросы на основе недостающих данных
- Исполнитель использует агрегированные данные для формирования ответа
- Оценщик учитывает наличие подтверждённых фактов из источников
- Стагнация определяется по улучшению оценки И наличию новых данных
"""

import json
import re
import logging
import time
from typing import TypedDict, List, Annotated, Literal, Optional, Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langgraph.checkpoint.memory import MemorySaver

# ============================================================================
# НАСТРОЙКА ЛОГГИРОВАНИЯ
# ============================================================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("AutonomousAgent")

# ============================================================================
# 1. ИНСТРУМЕНТЫ (улучшены: поиск с агрегацией нескольких запросов)
# ============================================================================

@tool
def search_docs(query: str) -> str:
    """Ищет информацию в локальной базе знаний (лекции курса)."""
    q = query.lower()
    if "rag" in q:
        return (
            "RAG (Retrieval-Augmented Generation) — подход, комбинирующий поиск по внешней базе знаний "
            "и генерацию текста. Это позволяет моделям отвечать точнее и актуальнее.\n"
            "В лекциях курса RAG описан как способ борьбы с галлюцинациями.\n"
            "Примеры: чат-боты для документов, поддержка клиентов, анализ юридических текстов, "
            "медицинские консультации на основе протоколов.\n"
            "Ключевое отличие от обычного ChatGPT: использует актуальные данные из внешних источников, "
            "а не только знания, заложенные при обучении.\n"
            "Цифры: точность ответов RAG на 20-30% выше, чем у обычной LLM без контекста."
        )
    elif "chatgpt" in q or "llm" in q:
        return (
            "Обычный ChatGPT — большая языковая модель (LLM), обученная на огромном корпусе текстов.\n"
            "Она генерирует ответы на основе своих внутренних знаний, но не имеет доступа к внешним данным "
            "в момент ответа (без плагинов).\n"
            "Может галлюцинировать, если вопрос выходит за пределы обучения.\n"
            "Области применения: общие консультации, написание текстов, программирование, обучение, "
            "развлечения, переводы.\n"
            "Цифры: ChatGPT справляется с 70% типовых запросов, но для специфических задач требует дообучения."
        )
    else:
        return "Информация не найдена."

web_search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """
    Выполняет поиск в интернете. Если результатов мало, использует локальную базу.
    Генерирует несколько вариантов запроса для полноты.
    """
    # Генерируем несколько вариантов запросов для разных аспектов
    queries = [
        query,
        query + " конкретные примеры цифры",
        query + " кейсы статистика",
        query + " реальные применения"
    ]
    # Объединяем результаты, удаляем дубликаты (упрощённо)
    all_results = []
    for q in queries[:2]:  # возьмём первые два, чтобы не перегружать
        try:
            res = web_search_tool.invoke(q)
            if res and len(res.strip()) > 50:
                all_results.append(res)
        except Exception as e:
            logger.warning(f"Ошибка поиска по запросу '{q}': {e}")
    if not all_results:
        logger.info("Результатов поиска нет, используем search_docs")
        return search_docs(query)
    combined = "\n\n".join(all_results)
    if len(combined) > 2000:
        combined = combined[:2000] + "... (обрезано)"
    return combined

@tool
def calculate(expression: str) -> str:
    """Безопасное вычисление математических выражений."""
    if not re.match(r'^[\d+\-*/().\s]+$', expression):
        return "Ошибка: недопустимые символы."
    try:
        result = eval(expression, {"__builtins__": {}})
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

tools = [search_docs, web_search, calculate]
tool_map = {t.name: t for t in tools}

# ============================================================================
# 2. НАСТРОЙКА LLM
# ============================================================================

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=1024)

# ============================================================================
# 3. МОДЕЛЬ ДЛЯ ОЦЕНКИ (Pydantic)
# ============================================================================

class EvaluationResult(BaseModel):
    score: int = Field(description="Оценка от 1 до 10 (10 – идеально)")
    feedback: str = Field(description="Развёрнутая обратная связь")
    is_satisfactory: bool = Field(description="true, если score >= 9")

# ============================================================================
# 4. СОСТОЯНИЕ (расширено: добавлены поля для агрегации и факт-чекинга)
# ============================================================================

class PlanExecuteState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    plan: List[Dict[str, Any]]
    current_step: int
    step_results: List[Dict[str, Any]]
    plan_status: Literal["active", "completed", "failed"]
    iteration: int
    max_iterations: int
    replan_count: int
    final_answer: Optional[str]
    retry_count: int
    max_retries: int
    eval_result: Optional[EvaluationResult]
    reflection_history: List[str]
    reflection_attempts: List[Dict]
    summary: str
    need_user_input: bool
    user_prompt: Optional[str]
    error: Optional[str]
    previous_eval: Optional[Dict]
    previous_final_answer: Optional[str]
    reflector_added_step: bool
    # Новые поля
    aggregated_data: str                # собранные данные из поиска
    fact_checked: bool                  # прошли ли проверку фактов
    sources: List[str]                  # ссылки на источники (если есть)
    data_sufficiency: int               # оценка достаточности данных (0-10)

# ============================================================================
# 5. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================================

def parse_plan(response_content: str) -> List[Dict[str, Any]]:
    try:
        json_match = re.search(r'\[.*\]', response_content, re.DOTALL)
        if json_match:
            plan = json.loads(json_match.group())
            if isinstance(plan, list) and all(isinstance(item, dict) for item in plan):
                for step in plan:
                    step.setdefault("tool", None)
                    step.setdefault("dependencies", [])
                return plan
    except Exception as e:
        logger.error(f"Ошибка парсинга плана: {e}")
    return [{"step_number": 1, "description": "Ответить на вопрос", "tool": None, "dependencies": []}]

def format_step_result(step_num: int, result: str) -> str:
    return f"Шаг {step_num}: {result}"

def update_summary(state: PlanExecuteState, step_result: dict) -> str:
    old_summary = state.get("summary", "")
    new_info = f"Шаг {step_result['step_number']}: {step_result['result'][:150]}..."
    if len(old_summary) + len(new_info) > 2000:
        old_summary = old_summary[-1500:]
    return old_summary + "\n" + new_info

def is_plan_valid(plan: List[Dict]) -> bool:
    numbers = [step.get("step_number") for step in plan]
    if len(numbers) != len(set(numbers)):
        return False
    for step in plan:
        for dep in step.get("dependencies", []):
            if dep not in numbers:
                return False
    return True

def detect_stagnation(state: PlanExecuteState) -> bool:
    """
    Стагнация: оценка не улучшается более 2 попыток ИЛИ данные не добавляются.
    """
    history = state.get("reflection_history", [])
    if len(history) < 2:
        return False
    scores = []
    for entry in history[-3:]:
        match = re.search(r'Оценка (\d+)/10', entry)
        if match:
            scores.append(int(match.group(1)))
    if len(scores) >= 2:
        # Если оценки одинаковы и <=8
        if len(set(scores)) == 1 and scores[0] <= 8:
            return True
        # Если последняя оценка не выше предыдущей (и уже было >=2 попыток)
        if len(scores) >= 2 and scores[-1] <= scores[-2] and len(history) >= 3:
            return True
    return False

def generate_specific_search_query(question: str, missing_aspect: str = "") -> str:
    """
    Генерирует конкретный поисковый запрос на основе вопроса и недостающего аспекта.
    """
    words = re.findall(r'\b\w{4,}\b', question)
    core = " ".join(words[:4])
    if missing_aspect:
        query = f"{core} {missing_aspect} конкретные примеры цифры статистика"
    else:
        query = f"{core} конкретные кейсы примеры цифры статистика"
    return query

def extract_facts_from_results(results: List[str]) -> str:
    """Извлекает факты (цифры, примеры) из результатов поиска."""
    combined = "\n".join(results)
    # Простейший эвристический сбор: строки, содержащие цифры или "пример"
    lines = combined.split('\n')
    facts = []
    for line in lines:
        if re.search(r'\d+%|\d+\.\d+|\d{4,}', line) or 'пример' in line.lower() or 'кейс' in line.lower():
            if len(line.strip()) > 20:
                facts.append(line.strip())
    if not facts:
        return combined[:500]
    return "\n".join(facts[:10])  # ограничим

# ============================================================================
# 6. УЗЛЫ ГРАФА (полностью переработаны)
# ============================================================================

def planner_node(state: PlanExecuteState) -> dict:
    logger.info("Планировщик: начало работы")
    if state.get("need_user_input"):
        return {"need_user_input": True}

    # Если уже есть план и мы перепланируем
    if state.get("plan") and state.get("replan_count", 0) > 0:
        plan = state["plan"]
        has_concrete = any("кейс" in step.get("description", "").lower() or "цифр" in step.get("description", "").lower()
                           for step in plan)
        if not has_concrete:
            topic = state["question"][:60]
            new_step = {
                "step_number": len(plan) + 1,
                "description": f"Поискать конкретные кейсы и цифры по теме: {topic}",
                "tool": "web_search",
                "dependencies": [state["current_step"]]
            }
            plan.append(new_step)
            logger.info(f"Перепланирование: добавлен шаг {new_step['step_number']}")
            return {"plan": plan, "replan_count": state.get("replan_count", 0) + 1}
        else:
            new_step = {
                "step_number": len(plan) + 1,
                "description": "Агрегировать найденные данные и проверить факты",
                "tool": None,
                "dependencies": [state["current_step"]]
            }
            plan.append(new_step)
            logger.info(f"Перепланирование: добавлен шаг {new_step['step_number']}")
            return {"plan": plan, "replan_count": state.get("replan_count", 0) + 1}

    system_prompt = """
Ты – планировщик. Разбей цель пользователя на чёткую последовательность шагов.
Каждый шаг должен быть конкретным и использовать инструмент, если необходимо.
Доступные инструменты: search_docs, web_search, calculate.

Формат ответа – JSON-массив, где каждый элемент:
{
  "step_number": номер,
  "description": краткое описание действия,
  "tool": имя инструмента или null,
  "dependencies": [номера шагов, от которых зависит]
}

Включай шаги для получения конкретных примеров и цифр.
Пример для "Сравни RAG и ChatGPT":
[
  {"step_number":1,"description":"Найти информацию о RAG (определение, принцип работы)","tool":"web_search","dependencies":[]},
  {"step_number":2,"description":"Найти информацию о ChatGPT (архитектура, обучение)","tool":"web_search","dependencies":[]},
  {"step_number":3,"description":"Сравнить архитектуру и подходы RAG и ChatGPT","tool":null,"dependencies":[1,2]},
  {"step_number":4,"description":"Найти конкретные кейсы и цифры использования RAG","tool":"web_search","dependencies":[1]},
  {"step_number":5,"description":"Найти конкретные кейсы и цифры использования ChatGPT","tool":"web_search","dependencies":[2]},
  {"step_number":6,"description":"Агрегировать и проверить факты","tool":null,"dependencies":[4,5]},
  {"step_number":7,"description":"Сравнить области применения с примерами и цифрами","tool":null,"dependencies":[6]},
  {"step_number":8,"description":"Сформулировать итоговый вывод с рекомендациями","tool":null,"dependencies":[3,7]}
]

Теперь сгенерируй план для задачи пользователя. Только JSON.
"""
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=state["question"])]
    try:
        response = llm.invoke(messages)
        plan = parse_plan(response.content)
        if not is_plan_valid(plan):
            logger.warning("Сгенерированный план невалиден. Запрашиваем уточнение.")
            return {
                "need_user_input": True,
                "user_prompt": "Сгенерированный план содержит ошибки. Уточните задачу.",
                "plan": []
            }
        logger.info(f"Сгенерирован план из {len(plan)} шагов.")
        for step in plan:
            logger.info(f"  Шаг {step['step_number']}: {step['description']} (tool: {step.get('tool', 'нет')})")
    except Exception as e:
        logger.error(f"Ошибка в планировщике: {e}")
        return {"error": str(e), "plan_status": "failed"}

    return {
        "plan": plan,
        "current_step": 0,
        "step_results": [],
        "plan_status": "active",
        "replan_count": 0,
        "need_user_input": False,
        "summary": "",
        "reflection_attempts": [],
        "reflector_added_step": False,
        "aggregated_data": "",
        "fact_checked": False,
        "sources": [],
        "data_sufficiency": 0
    }

def executor_node(state: PlanExecuteState) -> dict:
    logger.info("Исполнитель: начало выполнения шага")
    plan = state["plan"]
    current_idx = state["current_step"]
    if current_idx >= len(plan):
        return {"plan_status": "completed"}

    step = plan[current_idx]
    logger.info(f"Выполнение шага {step['step_number']}: {step['description']}")

    start_time = time.time()
    timeout = 30

    deps = step.get("dependencies", [])
    step_results = state.get("step_results", [])
    for dep in deps:
        dep_result = next((r for r in step_results if r["step_number"] == dep), None)
        if not dep_result or not dep_result.get("success", False):
            error_msg = f"Зависимость от шага {dep} не выполнена."
            logger.error(error_msg)
            result_entry = {
                "step_number": step["step_number"],
                "result": error_msg,
                "success": False,
                "error": error_msg
            }
            return {
                "step_results": state.get("step_results", []) + [result_entry],
                "current_step": current_idx + 1,
                "plan_status": "failed",
                "error": error_msg
            }

    tool_name = step.get("tool")
    result_text = ""
    success = True
    error = None

    # Если шаг — агрегация, используем отдельную логику
    if "агрегировать" in step["description"].lower() or "проверить факты" in step["description"].lower():
        # Собираем все результаты поиска
        search_results = []
        for res in state.get("step_results", []):
            if "web_search" in str(res.get("result", "")) or "найдено" in str(res.get("result", "")):
                search_results.append(res.get("result", ""))
        aggregated = extract_facts_from_results(search_results)
        # Проверяем наличие цифр
        digits = re.findall(r'\d+%|\d+\.\d+|\d{4,}', aggregated)
        if digits:
            sufficiency = min(10, len(digits) * 2)
        else:
            sufficiency = 3
        result_text = f"Агрегированные данные (факты, цифры, примеры):\n{aggregated}"
        # Сохраняем в состояние
        return {
            "step_results": state.get("step_results", []) + [{
                "step_number": step["step_number"],
                "result": result_text,
                "success": True,
                "error": None
            }],
            "current_step": current_idx + 1,
            "plan_status": "active" if current_idx + 1 < len(plan) else "completed",
            "aggregated_data": aggregated,
            "data_sufficiency": sufficiency,
            "fact_checked": True
        }

    if tool_name and tool_name in tool_map:
        try:
            query = step["description"]
            query = re.sub(r'(найти|искать|поиск|информацию|о|про|по теме)\s+', '', query, flags=re.IGNORECASE)
            if "rag" in query.lower() or "chatgpt" in query.lower():
                if "кейс" not in query.lower() and "пример" not in query.lower():
                    query = query + " кейсы примеры цифры"
            tool_result = tool_map[tool_name].invoke({"query": query})
            result_text = str(tool_result)
            if len(result_text) > 1500:
                result_text = result_text[:1500] + "... (обрезано)"
            logger.info(f"Инструмент {tool_name} вернул: {result_text[:150]}...")
        except Exception as e:
            error = str(e)
            success = False
            result_text = f"Ошибка инструмента: {e}"
            logger.error(f"Ошибка инструмента: {e}")
    else:
        # Шаг без инструмента – LLM, с использованием агрегированных данных
        context = state.get("summary", "")
        if not context:
            for r in state.get("step_results", []):
                context += format_step_result(r["step_number"], r["result"]) + "\n"

        # Используем агрегированные данные, если есть
        aggregated = state.get("aggregated_data", "")
        if aggregated:
            context += "\nАгрегированные факты и цифры:\n" + aggregated

        desc_lower = step["description"].lower()
        if "сравни" in desc_lower or "сравнить" in desc_lower:
            if "области" in desc_lower or "примен" in desc_lower:
                instruction = "Перечисли минимум 3 конкретные области применения для каждой технологии. Для каждой области приведи конкретные примеры и цифры из найденных данных. Если цифр нет, укажи это."
            else:
                instruction = "Выдели ключевые различия в виде пунктов. Для каждого пункта укажи, чем отличается RAG от ChatGPT. Используй найденные факты и цифры."
        elif "примен" in desc_lower or "использование" in desc_lower or "области" in desc_lower:
            instruction = "Перечисли минимум 3 конкретные области применения. Для каждой области приведи пример из реальной практики и цифры из источников."
        elif "вывод" in desc_lower or "итог" in desc_lower or "резюми" in desc_lower:
            instruction = "Сформулируй чёткий итоговый вывод, обобщи всё сказанное. Дай рекомендации, что лучше использовать в каких случаях. Приведи конкретные цифры и примеры из найденных данных."
        else:
            instruction = "Дай развёрнутый ответ, не повторяя предыдущие выводы. Старайся использовать конкретные примеры и цифры из источников."

        prompt = f"""
Ты – исполнитель, выполняешь конкретный шаг плана.

Текущий шаг: {step['description']}

Предыдущие результаты (резюме):
{context}

Задача: {instruction}

Важно: используй только те цифры и факты, которые есть в найденных данных. Не выдумывай.
Ответь кратко, но содержательно. Используй структуру (пункты, список).
"""
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            result_text = response.content.strip()
            logger.info(f"LLM сгенерировала: {result_text[:150]}...")
        except Exception as e:
            error = str(e)
            success = False
            result_text = f"Ошибка LLM: {e}"
            logger.error(f"Ошибка LLM: {e}")

    if time.time() - start_time > timeout:
        error = "Превышен таймаут выполнения шага"
        logger.warning(error)
        success = False
        result_text = error

    result_entry = {
        "step_number": step["step_number"],
        "result": result_text,
        "success": success,
        "error": error
    }
    new_results = state.get("step_results", []) + [result_entry]
    new_summary = update_summary(state, result_entry)

    new_idx = current_idx + 1
    if new_idx >= len(plan):
        plan_status = "completed"
        logger.info("Все шаги выполнены.")
    else:
        plan_status = "active"

    return {
        "step_results": new_results,
        "current_step": new_idx,
        "plan_status": plan_status,
        "summary": new_summary,
        "error": error
    }

def replanner_node(state: PlanExecuteState) -> dict:
    logger.info("Перепланирование из-за ошибки")
    plan = state["plan"]
    last_result = state["step_results"][-1] if state["step_results"] else None
    if not last_result or last_result.get("success", False):
        return {"plan_status": "active"}

    # Добавляем шаг поиска с конкретными ключевыми словами
    prev_step_num = last_result["step_number"]
    prev_step_desc = next((s["description"] for s in plan if s["step_number"] == prev_step_num), "")
    new_step = {
        "step_number": len(plan) + 1,
        "description": f"Найти дополнительную информацию по теме: {prev_step_desc[:80]} (кейсы, цифры, статистика)",
        "tool": "web_search",
        "dependencies": [prev_step_num]
    }
    plan.append(new_step)
    logger.info(f"Добавлен шаг {new_step['step_number']}: {new_step['description']}")
    return {
        "plan": plan,
        "plan_status": "active",
        "replan_count": state.get("replan_count", 0) + 1,
    }

def evaluator_node(state: PlanExecuteState) -> dict:
    logger.info("Оценщик: начало оценки")
    final_answer = state.get("final_answer")
    if not final_answer:
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage):
                final_answer = msg.content
                break
        if not final_answer and state["step_results"]:
            final_answer = state["step_results"][-1].get("result", "Ответ не сгенерирован")
        else:
            final_answer = "Ответ не сгенерирован"

    parser = PydanticOutputParser(pydantic_object=EvaluationResult)
    format_instructions = parser.get_format_instructions()

    prev_answer = state.get("previous_final_answer")
    comparison = ""
    if prev_answer:
        comparison = f"""
ПРЕДЫДУЩИЙ ОТВЕТ (для сравнения):
{prev_answer[:500]}

Сравни новый ответ с предыдущим. Если новый ответ улучшился (появились примеры, цифры, структура) – оценка должна быть выше.
Если улучшений нет – оценка не должна расти.
"""
    else:
        comparison = "Это первая оценка."

    # Учитываем наличие агрегированных данных
    agg_data = state.get("aggregated_data", "")
    data_suff = state.get("data_sufficiency", 0)
    data_note = f"Доступные данные: {len(agg_data)} символов, достаточность: {data_suff}/10."

    prompt = f"""
Ты – строгий, но справедливый критик. Оцени ответ по критериям:
1. Полнота: охвачены ли все аспекты вопроса?
2. Точность: есть ли фактические ошибки? (проверь по данным)
3. Ясность: легко ли понять ответ?
4. Наличие примеров и цифр: есть ли конкретные примеры, области применения, цифры из источников?
5. Структурированность: хорошо ли организован ответ?

Итоговая оценка – среднее арифметическое (округляй до целого).

Исходная цель: {state["question"]}
Результат агента: {final_answer}

{comparison}

Данные из источников: {data_note}

Оцени ответ строго. Если в ответе есть выдуманные цифры – снижай оценку.
Если примеры и цифры взяты из найденных данных – оценка выше.

{format_instructions}

is_satisfactory = true, только если итоговая оценка >= 9.
"""
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        eval_result = parser.parse(response.content)
        eval_result.is_satisfactory = eval_result.score >= 9
        logger.info(f"Оценка: {eval_result.score}/10")
        logger.info(f"Отзыв: {eval_result.feedback}")
    except Exception as e:
        logger.error(f"Ошибка парсинга оценки: {e}")
        eval_result = EvaluationResult(score=5, feedback="Не удалось распарсить оценку", is_satisfactory=False)

    return {
        "final_answer": final_answer,
        "retry_count": state.get("retry_count", 0),
        "eval_result": eval_result,
        "previous_eval": {"score": eval_result.score, "feedback": eval_result.feedback},
        "previous_final_answer": final_answer
    }

def reflector_node(state: PlanExecuteState) -> dict:
    logger.info("Рефлектор: начало")
    retry_count = state.get("retry_count", 0) + 1

    eval_result = state.get("eval_result")
    if not eval_result:
        return {"retry_count": retry_count}

    history = state.get("reflection_history", [])
    history.append(f"Попытка {retry_count}: Оценка {eval_result.score}/10. Отзыв: {eval_result.feedback}")

    attempts = state.get("reflection_attempts", [])
    attempts.append({
        "attempt": retry_count,
        "score": eval_result.score,
        "feedback": eval_result.feedback,
        "action": None
    })

    feedback_lower = eval_result.feedback.lower()
    need_more_data = any(phrase in feedback_lower for phrase in
                         ["не хватает", "дополнительная информация", "больше данных", "уточнить", "конкретные примеры", "цифры"])

    prev_score = state.get("previous_eval", {}).get("score")
    if prev_score is not None:
        improvement = eval_result.score - prev_score
        logger.info(f"Изменение оценки: {prev_score} -> {eval_result.score} ({'+' if improvement >=0 else ''}{improvement})")
    else:
        improvement = 0

    # Если оценка не улучшается или нужны данные, и реплан не исчерпан
    if (need_more_data or eval_result.score <= 8) and state.get("replan_count", 0) < 3:
        last_actions = [a.get("action") for a in attempts[-3:] if a.get("action") is not None]
        if "add_search" not in last_actions:
            # Генерируем конкретный запрос на основе недостающих данных
            missing = ""
            if "пример" in feedback_lower or "кейс" in feedback_lower:
                missing = "кейсы"
            if "цифр" in feedback_lower or "статистик" in feedback_lower:
                missing = "статистика цифры"
            if not missing:
                missing = "конкретные примеры и цифры"
            topic = state["question"]
            query = generate_specific_search_query(topic, missing)
            new_step = {
                "step_number": len(state["plan"]) + 1,
                "description": f"Поискать: {query}",
                "tool": "web_search",
                "dependencies": [state["current_step"]]
            }
            logger.info(f"Рефлектор добавляет новый шаг поиска: {new_step['description']}")
            plan = state["plan"] + [new_step]
            attempts[-1]["action"] = "add_search"
            return {
                "plan": plan,
                "current_step": len(plan) - 1,
                "retry_count": retry_count,
                "reflection_history": history,
                "reflection_attempts": attempts,
                "plan_status": "active",
                "replan_count": state.get("replan_count", 0) + 1,
                "reflector_added_step": True
            }
        elif "add_analysis" not in last_actions and state.get("replan_count", 0) < 3:
            # Добавляем шаг агрегации и проверки фактов
            new_step = {
                "step_number": len(state["plan"]) + 1,
                "description": "Агрегировать новые данные и проверить факты",
                "tool": None,
                "dependencies": [state["current_step"]]
            }
            logger.info(f"Рефлектор добавляет шаг агрегации: {new_step['description']}")
            plan = state["plan"] + [new_step]
            attempts[-1]["action"] = "add_analysis"
            return {
                "plan": plan,
                "current_step": len(plan) - 1,
                "retry_count": retry_count,
                "reflection_history": history,
                "reflection_attempts": attempts,
                "plan_status": "active",
                "reflector_added_step": True
            }

    # Если не добавляем шаги – перегенерируем ответ, используя агрегированные данные
    agg_data = state.get("aggregated_data", "")
    prompt = f"""
Ты – рефлектор. Улучши ответ, учитывая критику и используя агрегированные данные.

Исходная цель: {state["question"]}
Предыдущий ответ: {state["final_answer"]}
Обратная связь оценщика: {eval_result.feedback}

Доступные данные (факты, цифры, примеры):
{agg_data[:1000]}

Улучши ответ. Обязательно:
- Используй только те цифры и примеры, которые есть в данных.
- Сделай ответ структурированным.
- Добавь конкретные области применения с примерами.

Дай новый, более качественный ответ.
"""
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        new_answer = response.content.strip()
        logger.info(f"Сгенерирован улучшенный ответ (попытка {retry_count})")
    except Exception as e:
        logger.error(f"Ошибка в рефлекторе: {e}")
        return {"error": str(e)}
    attempts[-1]["action"] = "regenerate"
    return {
        "final_answer": new_answer,
        "retry_count": retry_count,
        "reflection_history": history,
        "reflection_attempts": attempts,
        "messages": [AIMessage(content=new_answer)],
        "previous_final_answer": state["final_answer"],
        "reflector_added_step": False
    }

def human_input_node(state: PlanExecuteState) -> dict:
    logger.info("Human-in-the-loop: запрос уточнения")
    prompt = state.get("user_prompt", "Пожалуйста, уточните задачу или предложите свой план.")
    print(f"\n❓ {prompt}")
    user_input = input("Ваш ответ: ")
    return {
        "need_user_input": False,
        "question": user_input,
        "plan": [],
        "plan_status": "active"
    }

# ============================================================================
# 7. МАРШРУТИЗАЦИЯ
# ============================================================================

def route_after_planner(state: PlanExecuteState) -> Literal["executor", "human_input", "finish"]:
    if state.get("need_user_input"):
        return "human_input"
    elif state.get("plan"):
        return "executor"
    else:
        return "finish"

def route_after_executor(state: PlanExecuteState) -> Literal["executor", "replanner", "evaluator", "finish"]:
    if state["plan_status"] == "completed":
        return "evaluator"
    elif state["plan_status"] == "failed":
        if state.get("replan_count", 0) < 3:
            return "replanner"
        else:
            return "finish"
    else:
        return "executor"

def route_after_evaluator(state: PlanExecuteState) -> Literal["reflector", "finish"]:
    eval_result = state.get("eval_result")
    retry_count = state.get("retry_count", 0)
    max_retries = state.get("max_retries", 4)

    if detect_stagnation(state):
        logger.warning("Обнаружена стагнация: оценка не улучшается. Завершаем.")
        return "finish"

    if eval_result and eval_result.is_satisfactory:
        logger.info("Оценка отличная, завершаем.")
        return "finish"
    elif retry_count < max_retries:
        logger.info("Оценка низкая, запускаем рефлексию.")
        return "reflector"
    else:
        logger.warning("Превышено число попыток, завершаем.")
        return "finish"

def route_after_reflector(state: PlanExecuteState) -> Literal["executor", "evaluator", "finish"]:
    if state.get("reflector_added_step", False) and state.get("plan_status") == "active":
        logger.info("Рефлектор добавил шаги, переходим к исполнителю.")
        return "executor"
    else:
        logger.info("Рефлектор не добавил шаги или план завершён, переходим к оценщику.")
        return "evaluator"

def route_after_human(state: PlanExecuteState) -> Literal["planner", "finish"]:
    if state.get("question"):
        return "planner"
    else:
        return "finish"

# ============================================================================
# 8. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(PlanExecuteState)
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)
builder.add_node("replanner", replanner_node)
builder.add_node("evaluator", evaluator_node)
builder.add_node("reflector", reflector_node)
builder.add_node("human_input", human_input_node)

builder.set_entry_point("planner")
builder.add_conditional_edges("planner", route_after_planner, {
    "executor": "executor",
    "human_input": "human_input",
    "finish": END
})
builder.add_conditional_edges("executor", route_after_executor, {
    "executor": "executor",
    "replanner": "replanner",
    "evaluator": "evaluator",
    "finish": END
})
builder.add_edge("replanner", "planner")
builder.add_conditional_edges("evaluator", route_after_evaluator, {
    "reflector": "reflector",
    "finish": END
})
builder.add_conditional_edges("reflector", route_after_reflector, {
    "executor": "executor",
    "evaluator": "evaluator",
    "finish": END
})
builder.add_conditional_edges("human_input", route_after_human, {
    "planner": "planner",
    "finish": END
})

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

# ============================================================================
# 9. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    question = "Сравни RAG и обычный ChatGPT. Опиши их основные различия и области применения."

    print("=" * 60)
    print(f"📝 Задача: {question}")
    print("=" * 60)

    thread_id = "test_session_001"
    config = {"configurable": {"thread_id": thread_id}}

    initial_state = {
        "messages": [HumanMessage(content=question)],
        "question": question,
        "plan": [],
        "current_step": 0,
        "step_results": [],
        "plan_status": "active",
        "iteration": 0,
        "max_iterations": 15,
        "replan_count": 0,
        "final_answer": None,
        "retry_count": 0,
        "max_retries": 4,
        "eval_result": None,
        "reflection_history": [],
        "reflection_attempts": [],
        "summary": "",
        "need_user_input": False,
        "user_prompt": None,
        "error": None,
        "previous_eval": None,
        "previous_final_answer": None,
        "reflector_added_step": False,
        "aggregated_data": "",
        "fact_checked": False,
        "sources": [],
        "data_sufficiency": 0
    }

    try:
        result = graph.invoke(initial_state, config)
    except Exception as e:
        logger.error(f"Ошибка выполнения: {e}")
        exit(1)

    print("\n" + "=" * 60)
    print("📊 ИТОГОВЫЙ ОТВЕТ:")
    final_answer = result.get("final_answer")
    if final_answer:
        print(final_answer)
    else:
        print("Ответ не сгенерирован")

    print(f"\n🔄 Попыток рефлексии: {result.get('retry_count', 0)}")
    eval_res = result.get("eval_result")
    if eval_res:
        print(f"📊 Итоговая оценка: {eval_res.score}/10")
        print(f"💬 Отзыв: {eval_res.feedback}")
    print(f"🏁 Статус плана: {result.get('plan_status')}")

    if result.get("reflection_history"):
        print("\n📜 История рефлексии:")
        for entry in result["reflection_history"]:
            print(f"  - {entry}")

    print("\n📋 Детали выполнения шагов:")
    for step in result.get("plan", []):
        step_res = next((r for r in result.get("step_results", []) if r["step_number"] == step["step_number"]), None)
        status = "✅" if step_res and step_res.get("success") else "❌" if step_res else "⏳"
        print(f"  Шаг {step['step_number']}: {step['description']} {status}")
        if step_res and step_res.get("result"):
            print(f"    Результат: {step_res['result'][:200]}...")
```

---

## Заключение

Поздравляем! Вместе с нами вы создали **полноценного автономного агента**, который умеет:

- **Планировать** – разбивать сложную задачу на шаги, используя LLM и генерируя структурированный план в JSON.
- **Выполнять** – вызывать внешние инструменты (поиск, вычисления) и генерировать текст.
- **Оценивать** – критиковать свой ответ, ставить оценку и давать обратную связь.
- **Рефлексировать** – улучшать ответ, при необходимости добавляя новые шаги для сбора недостающих данных (конкретных примеров и цифр).
- **Агрегировать данные** – собирать факты, цифры и примеры из найденных источников, проверять их достоверность.
- **Помнить** – сохранять состояние между сессиями благодаря `MemorySaver`.
- **Обрабатывать ошибки** – не зацикливаться, запрашивать уточнения у пользователя, обнаруживать стагнацию и логировать все действия.

Этот агент является **шагом к полностью самостоятельным системам**, способным работать без постоянного контроля человека. Вы теперь знакомы со всеми ключевыми паттернами современных ИИ‑агентов – от простого RAG до автономного планирования и рефлексии.

**Что дальше?**
- Экспериментируйте с разными моделями и инструментами.
- Добавляйте новые инструменты (базы данных, API, визуализацию).
- Встраивайте агента в реальные бизнес-процессы.
